# DPS-A position-wise independent discovery trend on TREC with Qwen3-8B-IT — wrapped prompt format

This notebook is a task-specific variant of the AGNews template. It independently discovers DPS-A-like features at each varied demonstration position for **TREC**, uses the wrapped Qwen chat prompt format, and saves compact per-feature summaries for the cross-task aggregation notebook.


In [1]:
import os

# Optional Hugging Face authentication.
# Recommended: set HF_TOKEN in your shell/Jupyter environment before running this notebook.
# If HF_TOKEN is already set, mirror it to the legacy variable used by some libraries.
if os.environ.get("HF_TOKEN") and not os.environ.get("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.environ["HF_TOKEN"]


In [2]:
import plotly.io as pio

# Change this to "jupyterlab", "notebook_connected", etc. if your environment needs it.
pio.renderers.default = "notebook"


## 1. Configuration

Set the task/model/prompt parameters here. `last_k=[-1]` means the final rendered prompt token, i.e. the terminal Qwen chat token such as `<|im_end|>`; `last_k=[-2]` selects the token immediately before it.


In [ ]:
# ============================================================
# User-editable configuration
# ============================================================

import os
import re
import json
import math
import random
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Sequence

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from IPython.display import display

import plotly.graph_objects as go
from plotly.subplots import make_subplots


# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42


def set_seed(seed: int) -> None:
    print("seed:", seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(SEED)


# -----------------------------
# Task config
# -----------------------------
TASK_KEY = 'trec'
TASK_DISPLAY_NAME = 'TREC'
DATASET_CANDIDATES = [('trec', None), ('CogComp/trec', None)]
LABEL_FIELD_CANDIDATES = ['coarse_label', 'label-coarse', 'label_coarse', 'label']
TEXT_FIELD_CANDIDATES = ['text', 'question']
INPUT_FIELD_NAME = 'Question'
OUTPUT_FIELD_NAME = 'Question Type'

TASK_INSTRUCTION = 'Pretend that you are an expert in question classification. For a given question, classify its coarse question type as one of: abbreviation, entity, description, human, location, or numeric.'

LABEL_TO_WORD = {0: 'Abbreviation', 1: 'Entity', 2: 'Description', 3: 'Human', 4: 'Location', 5: 'Numeric'}
WORD_TO_LABEL = {v: k for k, v in LABEL_TO_WORD.items()}
LABEL_IDS = list(LABEL_TO_WORD.keys())
NUM_LABELS = len(LABEL_IDS)


# -----------------------------
# Model / SAE config: Qwen3-8B-IT style chat usage
# -----------------------------
MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_SHORT_NAME = "Qwen3-8B-IT"
TARGET_LAYER = 18

PROMPT_FORMAT_NAME = "format2_wrapped_user_assistant_demos"
PROMPT_FORMAT_DESCRIPTION = (
    "Format 2: each demonstration is a user request followed by an assistant label; "
    "the final test query is also followed by an assistant gold-label message ending with the Qwen chat terminator."
)

# Adam Karvonen Qwen3 BatchTopK SAE release used by the Qwen3-8B notebook.
SAE_REPO_ID = "adamkarvonen/qwen3-8b-saes"
SAE_BASE_DIR = "saes_Qwen_Qwen3-8B_batch_top_k"
SAE_TRAINER = "trainer_0"
SAE_FILENAME = f"{SAE_BASE_DIR}/resid_post_layer_{TARGET_LAYER}/{SAE_TRAINER}/ae.pt"

DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Experiment config
# -----------------------------
N_TEST_QUERIES_PER_LABEL = 50
AUTO_REDUCE_N_TEST_QUERIES_PER_LABEL = True

NUM_DEMOS_TOTAL = 13              # M-shot prompt.
POSITIONS_TO_TEST = [1,2,3,4,5]         # None -> all positions 1..M. Or set e.g. [1, 2, 5, 7].

MAX_DEMO_CHARS = 1000            # Keep demos bounded; final query is not truncated by default.
MAX_QUERY_CHARS = None
USE_SAE_PREACTIVATIONS = True    # Use SAE pre-activations.

# Query-token measurement control.
# None -> use all tokens overlapping the final query text span.
# [-1] -> use the very last token in the fully rendered Qwen chat prompt.
#         With this wrapped prompt format, the prompt ends with assistant gold label + <|im_end|>,
#         so [-1] selects <|im_end|>, not the label word.
# [-2] -> selects the final assistant label token immediately before <|im_end|>.
last_k = [-1]
LAST_K = last_k
MEASURE_LAST_K_FROM_FULL_PROMPT = True

# Discovery/holdout split for avoiding circular evaluation.
DISCOVERY_FRACTION = 0.50

# Feature-set controls and plotting.
MAX_DPS_A_FEATURES_FOR_EVAL = None
TOP_K_FALLBACK_FEATURES = 50
RANDOM_CONTROL_SEED = 123

# Caching is very useful because the default position scan performs many model forwards.
USE_CACHE = True
FORCE_RECOMPUTE = False
CACHE_DIR = Path(f"./positionwise_independent_discovery_qwen3_8b_wrapped_prompt_cache/{TASK_KEY}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Summaries for cross-task aggregation.
SAVE_CROSS_TASK_ARTIFACTS = True
CROSS_TASK_SUMMARY_ROOT = Path("./dps_positionwise_wrapped_outputs")

# Progress / debugging
PRINT_EVERY = 100
DISPLAY_PROMPT_SNIPPET_CHARS = 3000

if POSITIONS_TO_TEST is None:
    POSITIONS_TO_TEST = list(range(1, int(NUM_DEMOS_TOTAL) + 1))
POSITIONS_TO_TEST = [int(p) for p in POSITIONS_TO_TEST]
POSITION_TO_IDX = {int(p): i for i, p in enumerate(POSITIONS_TO_TEST)}
IDX_TO_POSITION = {i: int(p) for p, i in POSITION_TO_IDX.items()}

assert NUM_DEMOS_TOTAL >= 1, "NUM_DEMOS_TOTAL must be at least 1."
assert N_TEST_QUERIES_PER_LABEL >= 1, "N_TEST_QUERIES_PER_LABEL must be positive."
assert len(POSITIONS_TO_TEST) >= 1, "POSITIONS_TO_TEST cannot be empty."
assert all(1 <= int(p) <= int(NUM_DEMOS_TOTAL) for p in POSITIONS_TO_TEST), POSITIONS_TO_TEST

if N_TEST_QUERIES_PER_LABEL > 1:
    DISCOVERY_N_PER_LABEL = int(round(float(DISCOVERY_FRACTION) * int(N_TEST_QUERIES_PER_LABEL)))
    DISCOVERY_N_PER_LABEL = max(1, min(int(N_TEST_QUERIES_PER_LABEL) - 1, DISCOVERY_N_PER_LABEL))
else:
    DISCOVERY_N_PER_LABEL = 1

print("TASK_KEY:", TASK_KEY)
print("TASK_DISPLAY_NAME:", TASK_DISPLAY_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("TARGET_LAYER:", TARGET_LAYER)
print("PROMPT_FORMAT_NAME:", PROMPT_FORMAT_NAME)
print("SAE:", SAE_REPO_ID, SAE_FILENAME)
print("N_TEST_QUERIES_PER_LABEL requested:", N_TEST_QUERIES_PER_LABEL)
print("NUM_DEMOS_TOTAL:", NUM_DEMOS_TOTAL)
print("POSITIONS_TO_TEST:", POSITIONS_TO_TEST)
print("LAST_K:", LAST_K)
print("MEASURE_LAST_K_FROM_FULL_PROMPT:", MEASURE_LAST_K_FROM_FULL_PROMPT)
print("USE_SAE_PREACTIVATIONS:", USE_SAE_PREACTIVATIONS)


## 2. Robust loaders and Qwen BatchTopK SAE wrapper


In [ ]:
# ============================================================
# Robust Hugging Face loaders + Qwen BatchTopK SAE wrapper
# ============================================================

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or None
if HF_TOKEN is None:
    print("Warning: HF_TOKEN is not set. Public resources may still load, but gated/rate-limited resources may fail.")
else:
    print("HF_TOKEN detected from environment.")


def hf_hub_download_robust(repo_id: str, filename: str, repo_type: Optional[str] = None) -> str:
    kwargs = {"repo_id": repo_id, "filename": filename}
    if repo_type is not None:
        kwargs["repo_type"] = repo_type
    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN
    try:
        return hf_hub_download(**kwargs)
    except TypeError:
        kwargs.pop("token", None)
        if HF_TOKEN is not None:
            kwargs["use_auth_token"] = HF_TOKEN
        return hf_hub_download(**kwargs)


def load_tokenizer_robust(model_name: str):
    kwargs = {
        "pretrained_model_name_or_path": model_name,
        "trust_remote_code": True,
        "use_fast": True,
    }
    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN
    try:
        tok = AutoTokenizer.from_pretrained(**kwargs)
    except TypeError:
        kwargs.pop("token", None)
        if HF_TOKEN is not None:
            kwargs["use_auth_token"] = HF_TOKEN
        tok = AutoTokenizer.from_pretrained(**kwargs)
    return tok


def load_model_robust(model_name: str):
    kwargs = {
        "pretrained_model_name_or_path": model_name,
        "torch_dtype": DTYPE,
        "trust_remote_code": True,
    }
    if torch.cuda.is_available():
        kwargs["device_map"] = "auto"
    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN
    try:
        mdl = AutoModelForCausalLM.from_pretrained(**kwargs)
    except TypeError:
        kwargs.pop("token", None)
        if HF_TOKEN is not None:
            kwargs["use_auth_token"] = HF_TOKEN
        mdl = AutoModelForCausalLM.from_pretrained(**kwargs)
    mdl.eval()
    try:
        mdl.config.use_cache = False
    except Exception:
        pass
    return mdl


def _call_load_dataset_with_fallback(path: str, config_name: Optional[str], split_spec):
    base_kwargs = {"path": path, "split": split_spec}
    if config_name is not None:
        base_kwargs["name"] = config_name
    if HF_TOKEN is not None:
        base_kwargs["token"] = HF_TOKEN

    attempts = []
    kw = dict(base_kwargs)
    kw["trust_remote_code"] = True
    attempts.append(kw)
    attempts.append(dict(base_kwargs))

    # Older datasets versions use use_auth_token rather than token.
    if HF_TOKEN is not None:
        kw = dict(base_kwargs)
        kw.pop("token", None)
        kw["use_auth_token"] = HF_TOKEN
        attempts.append(kw)

    last_error = None
    for kwargs in attempts:
        try:
            return load_dataset(**kwargs)
        except TypeError as e:
            last_error = e
            # Retry after removing arguments unsupported by older datasets.
            kwargs2 = dict(kwargs)
            kwargs2.pop("trust_remote_code", None)
            try:
                return load_dataset(**kwargs2)
            except Exception as e2:
                last_error = e2
        except Exception as e:
            last_error = e
    raise last_error


def load_task_splits():
    last_error = None
    for candidate in DATASET_CANDIDATES:
        if isinstance(candidate, str):
            path, config_name = candidate, None
        else:
            path, config_name = candidate
        try:
            print(f"Trying {TASK_DISPLAY_NAME} dataset: path={path!r}, config={config_name!r}")
            train_split, test_split = _call_load_dataset_with_fallback(path, config_name, ["train", "test"])
            print("Loaded dataset from:", path, "config:", config_name)
            return train_split, test_split, f"{path}" if config_name is None else f"{path}:{config_name}"
        except Exception as e:
            last_error = e
            print("  failed:", repr(e))
    raise RuntimeError(f"Failed to load {TASK_DISPLAY_NAME}. Last error: {last_error}")


class QwenBatchTopKSAE(torch.nn.Module):
    """Minimal wrapper for Adam Karvonen's Qwen3 BatchTopK SAE checkpoints."""

    def __init__(self, state_dict, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        if isinstance(state_dict, dict) and "model" in state_dict and isinstance(state_dict["model"], dict):
            state_dict = state_dict["model"]

        b_dec = state_dict["b_dec"].detach()
        b_enc = state_dict.get("encoder.bias", state_dict.get("b_enc")).detach()
        enc_w = state_dict.get("encoder.weight", state_dict.get("W_enc")).detach()
        dec_w = state_dict.get("decoder.weight", state_dict.get("W_dec")).detach()
        threshold = state_dict["threshold"]
        threshold = threshold.detach() if torch.is_tensor(threshold) else torch.tensor(threshold)
        k = state_dict.get("k", None)
        if torch.is_tensor(k):
            k = int(k.item())

        d_model = int(b_dec.numel())
        n_features = int(b_enc.numel())

        if enc_w.shape == (n_features, d_model):
            W_enc = enc_w.T.contiguous()
        elif enc_w.shape == (d_model, n_features):
            W_enc = enc_w.contiguous()
        else:
            raise ValueError(f"Unexpected encoder.weight shape {tuple(enc_w.shape)}")

        if dec_w.shape == (d_model, n_features):
            W_dec = dec_w.T.contiguous()
        elif dec_w.shape == (n_features, d_model):
            W_dec = dec_w.contiguous()
        else:
            raise ValueError(f"Unexpected decoder.weight shape {tuple(dec_w.shape)}")

        self.register_buffer("W_enc", W_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.to(device=self.device, dtype=self.dtype))
        self.k = k
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.b_dec

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, pre, torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))


def get_model_input_device(model) -> torch.device:
    return model.get_input_embeddings().weight.device


def get_layer_device(model, layer_idx: int) -> torch.device:
    block = model.get_submodule(f"model.layers.{layer_idx}")
    try:
        return next(block.parameters()).device
    except StopIteration:
        return get_model_input_device(model)


## 3. Load TREC, Qwen3-8B-IT, and the middle-layer SAE


In [ ]:
# ============================================================
# Load dataset, tokenizer, model, and SAE
# ============================================================

from datasets import load_dataset

# ---------------------------------------------------------------------
# Fix for newer `datasets` versions:
# legacy `load_dataset("trec")` can fail with:
#   "Dataset scripts are no longer supported, but found trec.py"
#
# For TREC, use the script-free SetFit/TREC-QC dataset and normalize
# its coarse labels into the canonical order used in this notebook:
#   0: Abbreviation
#   1: Entity
#   2: Description
#   3: Human
#   4: Location
#   5: Numeric
# ---------------------------------------------------------------------

HF_TOKEN = (
    globals().get("HF_TOKEN", None)
    or os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or None
)


def _load_dataset_robust_any(path, config_name=None, split=None):
    """Load a HF dataset while being compatible with token/use_auth_token variants."""
    base_kwargs = {"path": path}

    if config_name is not None:
        base_kwargs["name"] = config_name

    if split is not None:
        base_kwargs["split"] = split

    attempts = []

    # Preferred modern form.
    kwargs = dict(base_kwargs)
    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN
    attempts.append(kwargs)

    # Some older code paths accept trust_remote_code, but script-based
    # datasets may still fail. This is only an attempted fallback.
    kwargs = dict(base_kwargs)
    kwargs["trust_remote_code"] = True
    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN
    attempts.append(kwargs)

    # Older HF versions use use_auth_token.
    if HF_TOKEN is not None:
        kwargs = dict(base_kwargs)
        kwargs["use_auth_token"] = HF_TOKEN
        attempts.append(kwargs)

        kwargs = dict(base_kwargs)
        kwargs["trust_remote_code"] = True
        kwargs["use_auth_token"] = HF_TOKEN
        attempts.append(kwargs)

    last_error = None

    for kwargs in attempts:
        try:
            return load_dataset(**kwargs)
        except TypeError as e:
            last_error = e

            # Retry after dropping arguments unsupported by older versions.
            kwargs2 = dict(kwargs)
            kwargs2.pop("trust_remote_code", None)
            kwargs2.pop("token", None)

            try:
                return load_dataset(**kwargs2)
            except Exception as e2:
                last_error = e2

        except Exception as e:
            last_error = e

    raise last_error


def _load_train_test_splits_robust(path, config_name=None):
    """Return (train_split, test_split) from either split-list or DatasetDict loading."""
    split_error = None

    try:
        result = _load_dataset_robust_any(
            path=path,
            config_name=config_name,
            split=["train", "test"],
        )

        if isinstance(result, (list, tuple)) and len(result) == 2:
            return result[0], result[1]

    except Exception as e:
        split_error = e

    try:
        dataset_dict = _load_dataset_robust_any(
            path=path,
            config_name=config_name,
            split=None,
        )

        if "train" not in dataset_dict:
            raise KeyError(
                f"Dataset {path!r} did not contain a train split. "
                f"Available splits: {list(dataset_dict.keys())}"
            )

        if "test" in dataset_dict:
            return dataset_dict["train"], dataset_dict["test"]

        if "validation" in dataset_dict:
            print(
                f"Warning: dataset {path!r} has no test split; "
                "using validation as test."
            )
            return dataset_dict["train"], dataset_dict["validation"]

        raise KeyError(
            f"Dataset {path!r} did not contain a test or validation split. "
            f"Available splits: {list(dataset_dict.keys())}"
        )

    except Exception as e:
        if split_error is not None:
            raise RuntimeError(
                f"Failed split-list loading with: {repr(split_error)}; "
                f"then failed DatasetDict loading with: {repr(e)}"
            )

        raise


# Canonical TREC label order used by this notebook.
TREC_CANONICAL_SHORT_TO_ID = {
    "ABBR": 0,
    "ENTY": 1,
    "DESC": 2,
    "HUM": 3,
    "LOC": 4,
    "NUM": 5,
}

TREC_CANONICAL_LABEL_TO_WORD = {
    0: "Abbreviation",
    1: "Entity",
    2: "Description",
    3: "Human",
    4: "Location",
    5: "Numeric",
}

# SetFit/TREC-QC label_coarse integer order in the dataset viewer:
#   DESC=0, ENTY=1, ABBR=2, HUM=3, NUM=4, LOC=5
# Convert it into the notebook's canonical order:
#   ABBR=0, ENTY=1, DESC=2, HUM=3, LOC=4, NUM=5
SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID = {
    0: 2,  # DESC -> Description
    1: 1,  # ENTY -> Entity
    2: 0,  # ABBR -> Abbreviation
    3: 3,  # HUM  -> Human
    4: 5,  # NUM  -> Numeric
    5: 4,  # LOC  -> Location
}

SETFIT_TREC_COARSE_TEXT_TO_SHORT = {
    "description and abstract concepts": "DESC",
    "entities": "ENTY",
    "abbreviation": "ABBR",
    "human beings": "HUM",
    "locations": "LOC",
    "numeric values": "NUM",
}


def _canonicalize_setfit_trec_example(example):
    """Add canonical `coarse_label`, canonical `label`, and `question` columns."""
    canonical_id = None

    # Most reliable field in SetFit/TREC-QC.
    raw_short = example.get("label_coarse_original", None)
    if raw_short is not None:
        raw_short = str(raw_short).strip().upper()
        if raw_short in TREC_CANONICAL_SHORT_TO_ID:
            canonical_id = TREC_CANONICAL_SHORT_TO_ID[raw_short]

    # Fallback through readable coarse text.
    if canonical_id is None:
        raw_text = example.get("label_coarse_text", None)
        if raw_text is not None:
            raw_text = str(raw_text).strip().lower()
            if raw_text in SETFIT_TREC_COARSE_TEXT_TO_SHORT:
                short = SETFIT_TREC_COARSE_TEXT_TO_SHORT[raw_text]
                canonical_id = TREC_CANONICAL_SHORT_TO_ID[short]

    # Fallback through SetFit's integer coarse-label order.
    if canonical_id is None:
        raw_id = example.get("label_coarse", None)
        if raw_id is not None:
            raw_id = int(raw_id)
            if raw_id in SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID:
                canonical_id = SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID[raw_id]

    if canonical_id is None:
        raise ValueError(
            "Could not canonicalize TREC label for example with keys: "
            f"{list(example.keys())}"
        )

    text = example.get("text", None)
    if text is None:
        text = example.get("question", None)

    if text is None:
        raise ValueError(
            "Could not find TREC question text. Available keys: "
            f"{list(example.keys())}"
        )

    return {
        "question": str(text),
        "text": str(text),
        "coarse_label": int(canonical_id),
        "label": int(canonical_id),
    }


def _normalize_setfit_trec_qc_split(split):
    """Normalize SetFit/TREC-QC columns to the schema expected downstream."""
    return split.map(
        _canonicalize_setfit_trec_example,
        desc="Canonicalizing SetFit/TREC-QC coarse labels",
    )


def _load_trec_splits_script_free():
    """Load TREC without relying on the deprecated/unsupported legacy trec.py script."""
    global LABEL_TO_WORD, WORD_TO_LABEL, LABEL_IDS, NUM_LABELS
    global LABEL_FIELD_CANDIDATES, TEXT_FIELD_CANDIDATES, DATASET_CANDIDATES

    # Ensure the rest of the notebook uses the canonical TREC label order.
    LABEL_TO_WORD = dict(TREC_CANONICAL_LABEL_TO_WORD)
    WORD_TO_LABEL = {v: k for k, v in LABEL_TO_WORD.items()}
    LABEL_IDS = list(LABEL_TO_WORD.keys())
    NUM_LABELS = len(LABEL_IDS)

    # Make sure downstream helper functions pick the canonicalized column first.
    LABEL_FIELD_CANDIDATES = [
        "coarse_label",
        "label",
        "label_coarse",
        "coarse",
        "label-coarse",
    ]

    TEXT_FIELD_CANDIDATES = [
        "question",
        "text",
    ]

    # Prefer script-free SetFit/TREC-QC.
    # Keep old names as last-resort fallbacks, but they may fail under new datasets.
    DATASET_CANDIDATES = [
        ("SetFit/TREC-QC", None),
        ("CogComp/trec", None),
        ("trec", None),
    ]

    last_error = None

    for path, config_name in DATASET_CANDIDATES:
        try:
            print(
                f"Trying {TASK_DISPLAY_NAME} dataset: "
                f"path={path!r}, config={config_name!r}"
            )

            train_split, test_split = _load_train_test_splits_robust(
                path=path,
                config_name=config_name,
            )

            if path == "SetFit/TREC-QC":
                train_split = _normalize_setfit_trec_qc_split(train_split)
                test_split = _normalize_setfit_trec_qc_split(test_split)

            print("Loaded dataset from:", path, "config:", config_name)

            return (
                train_split,
                test_split,
                f"{path}" if config_name is None else f"{path}:{config_name}",
            )

        except Exception as e:
            last_error = e
            print("  failed:", repr(e))

    raise RuntimeError(
        f"Failed to load {TASK_DISPLAY_NAME}. Last error: {last_error}"
    )


def load_task_splits_fixed():
    """Drop-in fixed replacement for load_task_splits()."""
    is_trec = (
        str(globals().get("TASK_KEY", "")).lower() == "trec"
        or str(globals().get("TASK_DISPLAY_NAME", "")).lower() == "trec"
    )

    if is_trec:
        return _load_trec_splits_script_free()

    last_error = None

    for candidate in DATASET_CANDIDATES:
        if isinstance(candidate, str):
            path, config_name = candidate, None
        else:
            path, config_name = candidate

        try:
            print(
                f"Trying {TASK_DISPLAY_NAME} dataset: "
                f"path={path!r}, config={config_name!r}"
            )

            train_split, test_split = _load_train_test_splits_robust(
                path=path,
                config_name=config_name,
            )

            print("Loaded dataset from:", path, "config:", config_name)

            return (
                train_split,
                test_split,
                f"{path}" if config_name is None else f"{path}:{config_name}",
            )

        except Exception as e:
            last_error = e
            print("  failed:", repr(e))

    raise RuntimeError(
        f"Failed to load {TASK_DISPLAY_NAME}. Last error: {last_error}"
    )


# Override the previous buggy/legacy loader.
load_task_splits = load_task_splits_fixed


# -----------------------------
# Load dataset
# -----------------------------
train_data, test_data, DATASET_NAME = load_task_splits()

print("Loaded task:", TASK_DISPLAY_NAME)
print("Dataset:", DATASET_NAME)
print("Train size:", len(train_data))
print("Test size:", len(test_data))
print("Train columns:", getattr(train_data, "column_names", None))
print("Test columns:", getattr(test_data, "column_names", None))

if str(globals().get("TASK_KEY", "")).lower() == "trec":
    print("Canonical TREC label mapping:")
    for k, v in LABEL_TO_WORD.items():
        print(f"  {k}: {v}")

    try:
        print("Train coarse-label counts:")
        display(
            pd.Series(train_data["coarse_label"])
            .value_counts()
            .sort_index()
            .rename(index=LABEL_TO_WORD)
            .to_frame("count")
        )

        print("Test coarse-label counts:")
        display(
            pd.Series(test_data["coarse_label"])
            .value_counts()
            .sort_index()
            .rename(index=LABEL_TO_WORD)
            .to_frame("count")
        )
    except Exception as e:
        print("Could not display TREC label-count sanity check:", repr(e))


# -----------------------------
# Load tokenizer
# -----------------------------
print("Loading tokenizer...")
tokenizer = load_tokenizer_robust(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


# -----------------------------
# Load model
# -----------------------------
print("Loading model...")
model = load_model_robust(MODEL_NAME)

MODEL_INPUT_DEVICE = get_model_input_device(model)
TARGET_LAYER_DEVICE = get_layer_device(model, TARGET_LAYER)

print("MODEL_INPUT_DEVICE:", MODEL_INPUT_DEVICE)
print("TARGET_LAYER_DEVICE:", TARGET_LAYER_DEVICE)


# -----------------------------
# Load SAE
# -----------------------------
print("Loading SAE...")
sae_path = hf_hub_download_robust(SAE_REPO_ID, SAE_FILENAME)

try:
    sae_state = torch.load(
        sae_path,
        map_location="cpu",
        weights_only=True,
    )
except TypeError:
    sae_state = torch.load(
        sae_path,
        map_location="cpu",
    )

sae = QwenBatchTopKSAE(
    sae_state,
    device=TARGET_LAYER_DEVICE,
    dtype=DTYPE,
)

sae.eval()

print(
    f"Loaded SAE: d_model={sae.d_model}, "
    f"n_features={sae.n_features}, k={sae.k}"
)

## 4. Prompt construction and position-scan prompt plan

The fixed context is sampled normally for every position-wise DPS search. No prefix-label exclusion or “first occurrence” constraint is imposed.


In [ ]:
# # ============================================================
# # Prompt construction and position-scan sampling plan
# # ============================================================

# QUERY_START_SENTINEL = "ZXQ_QUERY_START_4f4cc908"
# QUERY_END_SENTINEL = "ZXQ_QUERY_END_4f4cc908"


# # -----------------------------
# # LAST_K helpers
# # -----------------------------

# def normalize_last_k_spec(last_k=LAST_K):
#     if last_k is None:
#         return None
#     if isinstance(last_k, (int, np.integer)):
#         k = int(last_k)
#         if k <= 0:
#             raise ValueError(f"Integer LAST_K must be positive, got {last_k}.")
#         return list(range(-k, 0))
#     vals = [int(x) for x in list(last_k)]
#     if len(vals) == 0:
#         raise ValueError("LAST_K must be None, a positive integer, or a non-empty list of integer offsets.")
#     return vals


# def measurement_selection_slug(last_k=LAST_K) -> str:
#     if last_k is None:
#         return "all_query_text_tokens"
#     if isinstance(last_k, (int, np.integer)):
#         return f"full_prompt_last_{int(last_k)}_tokens"
#     vals = [int(x) for x in list(last_k)]
#     def _fmt(v):
#         return f"m{abs(v)}" if v < 0 else f"p{v}"
#     return "full_prompt_offsets_" + "_".join(_fmt(v) for v in vals)


# def selected_query_region_description() -> str:
#     if LAST_K is None:
#         return "all tokens overlapping the final query text span"
#     return (
#         f"token offsets {normalize_last_k_spec(LAST_K)} from the end of the full rendered prompt token sequence "
#         f"(MEASURE_LAST_K_FROM_FULL_PROMPT={MEASURE_LAST_K_FROM_FULL_PROMPT}). "
#         "With the default LAST_K=[-1], this should select the terminal chat token, e.g. <|im_end|>, "
#         "rather than the final label word. LAST_K=[-2] selects the token immediately before <|im_end|>."
#     )


# # -----------------------------
# # Generic task text / label helpers
# # -----------------------------

# def _has_key(example: Dict[str, Any], key: str) -> bool:
#     try:
#         return key in example.keys()
#     except Exception:
#         return key in example


# def get_label_id(example: Dict[str, Any]) -> int:
#     for key in LABEL_FIELD_CANDIDATES:
#         if _has_key(example, key):
#             raw = int(example[key])
#             if raw in LABEL_TO_WORD:
#                 return raw
#             # Some datasets store labels as 1..K instead of 0..K-1.
#             if (raw - 1) in LABEL_TO_WORD:
#                 return raw - 1
#             raise ValueError(f"Label value {raw} from field {key!r} is not compatible with LABEL_TO_WORD={LABEL_TO_WORD}.")
#     raise KeyError(f"Could not find any label field among {LABEL_FIELD_CANDIDATES}. Example keys={list(example.keys())}")


# def clean_task_text(text: str, max_chars: Optional[int] = MAX_DEMO_CHARS) -> str:
#     text = str(text).replace("\n", " ").replace("\t", " ").strip()
#     text = re.sub(r"\s+", " ", text)
#     if max_chars is not None and len(text) > int(max_chars):
#         text = text[: int(max_chars)].rstrip() + "..."
#     return text


# def get_example_text(example: Dict[str, Any], max_chars: Optional[int] = MAX_DEMO_CHARS) -> str:
#     # Yahoo Answers benefits from preserving its multi-field structure.
#     if TASK_KEY == "yahoo":
#         pieces = []
#         field_labels = {
#             "question_title": "Title",
#             "question_content": "Question details",
#             "best_answer": "Best answer",
#             "text": "Text",
#         }
#         for key in TEXT_FIELD_CANDIDATES:
#             if _has_key(example, key):
#                 val = str(example[key]).strip()
#                 if val and val.lower() != "none":
#                     pieces.append(f"{field_labels.get(key, key)}: {val}")
#         if not pieces:
#             raise KeyError(f"Could not construct Yahoo text from keys={list(example.keys())}")
#         return clean_task_text("\n".join(pieces), max_chars=max_chars)

#     for key in TEXT_FIELD_CANDIDATES:
#         if _has_key(example, key):
#             return clean_task_text(example[key], max_chars=max_chars)
#     raise KeyError(f"Could not find text field among {TEXT_FIELD_CANDIDATES}. Example keys={list(example.keys())}")


# def label_word_for_example(example: Dict[str, Any]) -> str:
#     return LABEL_TO_WORD[int(get_label_id(example))]


# def format_demo_text_block(example: Dict[str, Any], idx: int) -> str:
#     text = get_example_text(example, max_chars=MAX_DEMO_CHARS)
#     label = label_word_for_example(example)
#     return f"Example {idx}\n{INPUT_FIELD_NAME}:\n{text}\n{OUTPUT_FIELD_NAME}:\n{label}"


# def format_user_request(example: Dict[str, Any], idx: int, *, mark_text: bool) -> str:
#     max_chars = MAX_QUERY_CHARS if mark_text else MAX_DEMO_CHARS
#     text = get_example_text(example, max_chars=max_chars)
#     if mark_text:
#         text = f"{QUERY_START_SENTINEL}{text}{QUERY_END_SENTINEL}"
#     return f"Example {idx}\n{INPUT_FIELD_NAME}:\n{text}\n{OUTPUT_FIELD_NAME}:"


# def format_assistant_label(example: Dict[str, Any]) -> str:
#     return label_word_for_example(example)


# # -----------------------------
# # Qwen chat rendering
# # -----------------------------

# def manual_qwen_chat_template(messages: List[Dict[str, str]]) -> str:
#     """Fallback ChatML renderer matching the Qwen-style <|im_start|>/<|im_end|> convention."""
#     parts = []
#     for msg in messages:
#         role = str(msg["role"])
#         content = str(msg["content"])
#         parts.append(f"<|im_start|>{role}\n{content}<|im_end|>\n")
#     return "".join(parts).rstrip()


# def apply_qwen_chat_template(messages: List[Dict[str, str]]) -> str:
#     try:
#         return tokenizer.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=False,
#             enable_thinking=False,
#         )
#     except TypeError:
#         try:
#             return tokenizer.apply_chat_template(
#                 messages,
#                 tokenize=False,
#                 add_generation_prompt=False,
#             )
#         except Exception:
#             return manual_qwen_chat_template(messages)
#     except Exception:
#         return manual_qwen_chat_template(messages)


# def render_prompt_from_messages_with_query_span(messages: List[Dict[str, str]]) -> Tuple[str, Tuple[int, int]]:
#     rendered = apply_qwen_chat_template(messages)
#     s = rendered.find(QUERY_START_SENTINEL)
#     e = rendered.find(QUERY_END_SENTINEL)
#     if s < 0 or e < 0 or e <= s:
#         tail = rendered[:1000] + "\n...\n" + rendered[-1000:]
#         raise ValueError(f"Could not locate query sentinels in rendered prompt. Excerpt:\n{tail}")

#     query_text_start_marked = s + len(QUERY_START_SENTINEL)
#     query_text = rendered[query_text_start_marked:e]

#     # Remove sentinels while preserving the clean span of the final query text.
#     prompt = rendered[:s] + query_text + rendered[e + len(QUERY_END_SENTINEL):]
#     query_char_span = (s, s + len(query_text))
#     return prompt.rstrip(), query_char_span


# # -----------------------------
# # Prompt-format builders
# # -----------------------------

# def build_prompt_format1_system_demos(
#     demos: List[Dict[str, Any]],
#     query_example: Dict[str, Any],
# ) -> Tuple[str, Tuple[int, int]]:
#     """Format 1: all demonstrations in the system prompt; final query in user; gold label in assistant."""
#     demo_blocks = [format_demo_text_block(d, i + 1) for i, d in enumerate(demos)]
#     system_sections = [TASK_INSTRUCTION]
#     if len(demo_blocks) > 0:
#         system_sections.append("Demonstrations:\n\n" + "\n\n".join(demo_blocks))
#     system_content = "\n\n".join(system_sections)

#     user_content = format_user_request(
#         query_example,
#         len(demos) + 1,
#         mark_text=True,
#     )

#     messages = [
#         {"role": "system", "content": system_content},
#         {"role": "user", "content": user_content},
#         {"role": "assistant", "content": format_assistant_label(query_example)},
#     ]
#     return render_prompt_from_messages_with_query_span(messages)


# def build_prompt_format2_wrapped_demos(
#     demos: List[Dict[str, Any]],
#     query_example: Dict[str, Any],
# ) -> Tuple[str, Tuple[int, int]]:
#     """Format 2: each demonstration is represented as a user request plus assistant label."""
#     messages = [{"role": "system", "content": TASK_INSTRUCTION}]

#     for i, demo in enumerate(demos, start=1):
#         messages.append({
#             "role": "user",
#             "content": format_user_request(demo, i, mark_text=False),
#         })
#         messages.append({
#             "role": "assistant",
#             "content": format_assistant_label(demo),
#         })

#     messages.append({
#         "role": "user",
#         "content": format_user_request(query_example, len(demos) + 1, mark_text=True),
#     })
#     messages.append({
#         "role": "assistant",
#         "content": format_assistant_label(query_example),
#     })

#     return render_prompt_from_messages_with_query_span(messages)


# def build_prompt(demos: List[Dict[str, Any]], query_example: Dict[str, Any]) -> Tuple[str, Tuple[int, int]]:
#     if PROMPT_FORMAT_NAME == "format1_system_demos_user_query_assistant_label":
#         return build_prompt_format1_system_demos(demos, query_example)
#     if PROMPT_FORMAT_NAME == "format2_wrapped_user_assistant_demos":
#         return build_prompt_format2_wrapped_demos(demos, query_example)
#     raise ValueError(f"Unknown PROMPT_FORMAT_NAME: {PROMPT_FORMAT_NAME}")


# # -----------------------------
# # Position-scan sampling plan
# # -----------------------------
# # No prefix/first-encounter constraint is applied in this notebook.
# # The fixed context is sampled once per test query and reused across varied positions.

# def index_by_label(dataset) -> Dict[int, List[int]]:
#     by_label = {int(l): [] for l in LABEL_IDS}
#     for i, row in enumerate(dataset):
#         label_id = int(get_label_id(row))
#         if label_id in by_label:
#             by_label[label_id].append(int(i))
#     return by_label


# train_by_label = index_by_label(train_data)
# test_by_label = index_by_label(test_data)

# print("Train label counts:", {LABEL_TO_WORD[k]: len(v) for k, v in train_by_label.items()})
# print("Test label counts:", {LABEL_TO_WORD[k]: len(v) for k, v in test_by_label.items()})

# min_test_count = min(len(test_by_label[int(l)]) for l in LABEL_IDS)
# if int(N_TEST_QUERIES_PER_LABEL) > int(min_test_count):
#     if AUTO_REDUCE_N_TEST_QUERIES_PER_LABEL:
#         print(
#             f"Warning: requested N_TEST_QUERIES_PER_LABEL={N_TEST_QUERIES_PER_LABEL}, "
#             f"but the smallest test class has only {min_test_count} examples. "
#             f"Reducing N_TEST_QUERIES_PER_LABEL to {min_test_count}."
#         )
#         N_TEST_QUERIES_PER_LABEL = int(min_test_count)
#     else:
#         raise ValueError(
#             f"Not enough test examples for all labels: min={min_test_count}, requested={N_TEST_QUERIES_PER_LABEL}."
#         )

# if N_TEST_QUERIES_PER_LABEL > 1:
#     DISCOVERY_N_PER_LABEL = int(round(float(DISCOVERY_FRACTION) * int(N_TEST_QUERIES_PER_LABEL)))
#     DISCOVERY_N_PER_LABEL = max(1, min(int(N_TEST_QUERIES_PER_LABEL) - 1, DISCOVERY_N_PER_LABEL))
# else:
#     DISCOVERY_N_PER_LABEL = 1


# def balanced_label_counts(total: int, labels: Sequence[int], seed: int) -> Dict[int, int]:
#     """Distribute total examples across labels as uniformly as possible."""
#     labels = list(map(int, labels))
#     base = int(total) // len(labels)
#     remainder = int(total) % len(labels)
#     rng = random.Random(int(seed))
#     shuffled = labels[:]
#     rng.shuffle(shuffled)
#     counts = {int(l): int(base) for l in labels}
#     for l in shuffled[:remainder]:
#         counts[int(l)] += 1
#     return counts


# def sample_fixed_test_query_indices(per_label: int = None, seed: int = SEED) -> pd.DataFrame:
#     if per_label is None:
#         per_label = int(N_TEST_QUERIES_PER_LABEL)
#     rng = random.Random(int(seed))
#     rows = []
#     for label_id in LABEL_IDS:
#         pool = list(test_by_label[int(label_id)])
#         if len(pool) < int(per_label):
#             raise ValueError(f"Not enough {TASK_DISPLAY_NAME} test examples for label {label_id}: {len(pool)} < {per_label}")
#         chosen = rng.sample(pool, int(per_label))
#         for q_pos, test_index in enumerate(chosen):
#             split = "discovery" if int(q_pos) < int(DISCOVERY_N_PER_LABEL) else "holdout"
#             rows.append({
#                 "query_global_id": len(rows),
#                 "query_label_id": int(label_id),
#                 "query_label": LABEL_TO_WORD[int(label_id)],
#                 "query_index_within_label": int(q_pos),
#                 "test_index": int(test_index),
#                 "split": split,
#             })
#     return pd.DataFrame(rows)


# def sample_fixed_context_pool(query_global_id: int) -> Tuple[int, ...]:
#     """Sample the M-1 fixed demonstrations reused across all position scans for one query.

#     No prefix-label exclusion constraint is applied. For a position-k DPS search,
#     labels in positions 1...(k-1) are sampled normally and may include the test-query label.
#     """
#     total_fixed = int(NUM_DEMOS_TOTAL) - 1
#     counts = balanced_label_counts(
#         total_fixed,
#         LABEL_IDS,
#         seed=SEED + 17_000 + 101 * int(query_global_id),
#     )
#     rng = random.Random(SEED + 31_000 + 1_003 * int(query_global_id))
#     fixed_indices = []
#     for label_id in LABEL_IDS:
#         c = counts[int(label_id)]
#         if c <= 0:
#             continue
#         pool = list(train_by_label[int(label_id)])
#         if len(pool) < c:
#             raise ValueError(f"Not enough train demos for label {label_id}: {len(pool)} < {c}")
#         fixed_indices.extend(rng.sample(pool, c))
#     rng.shuffle(fixed_indices)
#     assert len(fixed_indices) == total_fixed
#     return tuple(map(int, fixed_indices))


# def sample_variable_demo_indices_by_label(query_global_id: int, fixed_context_indices: Sequence[int]) -> Dict[int, int]:
#     """Sample one variable demonstration per label, reused across all varied positions for one query."""
#     exclude = set(map(int, fixed_context_indices))
#     out = {}
#     for label_id in LABEL_IDS:
#         pool = [int(i) for i in train_by_label[int(label_id)] if int(i) not in exclude]
#         if len(pool) == 0:
#             raise ValueError(f"No available variable demos for label {label_id} after exclusions.")
#         rng = random.Random(SEED + 59_000 + 10_007 * int(query_global_id) + int(label_id))
#         out[int(label_id)] = int(rng.choice(pool))
#     return out


# def insert_variable_at_position(fixed_context_indices: Sequence[int], variable_idx: int, varied_position: int) -> Tuple[int, ...]:
#     """Insert variable_idx at 1-indexed varied_position into the fixed M-1 context pool."""
#     fixed = list(map(int, fixed_context_indices))
#     p0 = int(varied_position) - 1
#     if p0 < 0 or p0 > len(fixed):
#         raise IndexError(f"varied_position must be in 1..{len(fixed) + 1}, got {varied_position}")
#     return tuple(fixed[:p0] + [int(variable_idx)] + fixed[p0:])


# def build_position_scan_plan() -> pd.DataFrame:
#     query_df = sample_fixed_test_query_indices()
#     rows = []
#     for r in query_df.itertuples(index=False):
#         fixed_context_indices = sample_fixed_context_pool(query_global_id=int(r.query_global_id))
#         variable_by_label = sample_variable_demo_indices_by_label(
#             query_global_id=int(r.query_global_id),
#             fixed_context_indices=fixed_context_indices,
#         )

#         for varied_position in POSITIONS_TO_TEST:
#             p0 = int(varied_position) - 1
#             for varied_label_id in LABEL_IDS:
#                 variable_idx = int(variable_by_label[int(varied_label_id)])
#                 demo_indices = insert_variable_at_position(
#                     fixed_context_indices=fixed_context_indices,
#                     variable_idx=variable_idx,
#                     varied_position=int(varied_position),
#                 )
#                 non_varied_demo_indices = tuple(demo_indices[:p0] + demo_indices[p0 + 1:])
#                 rows.append({
#                     "query_global_id": int(r.query_global_id),
#                     "query_label_id": int(r.query_label_id),
#                     "query_label": str(r.query_label),
#                     "query_index_within_label": int(r.query_index_within_label),
#                     "test_index": int(r.test_index),
#                     "split": str(r.split),
#                     "varied_position": int(varied_position),
#                     "varied_label_id": int(varied_label_id),
#                     "varied_label": LABEL_TO_WORD[int(varied_label_id)],
#                     "variable_demo_index": int(variable_idx),
#                     "fixed_context_indices": tuple(fixed_context_indices),
#                     "non_varied_demo_indices": tuple(non_varied_demo_indices),
#                     "demo_indices": tuple(demo_indices),
#                 })
#     return pd.DataFrame(rows)


# position_scan_plan_df = build_position_scan_plan()

# # Structural sanity checks.
# expected_queries = NUM_LABELS * int(N_TEST_QUERIES_PER_LABEL)
# expected_prompts = expected_queries * len(POSITIONS_TO_TEST) * NUM_LABELS
# assert len(position_scan_plan_df) == expected_prompts, (len(position_scan_plan_df), expected_prompts)
# assert position_scan_plan_df["query_global_id"].nunique() == expected_queries
# assert position_scan_plan_df.groupby(["query_global_id", "varied_position"]).size().eq(NUM_LABELS).all()
# assert position_scan_plan_df.groupby(["query_global_id", "varied_position"])["non_varied_demo_indices"].nunique().eq(1).all()
# assert position_scan_plan_df.groupby(["query_global_id", "varied_label_id"])["variable_demo_index"].nunique().eq(1).all()

# print("Task:", TASK_DISPLAY_NAME)
# print("Dataset:", DATASET_NAME)
# print("Prompt format:", PROMPT_FORMAT_NAME)
# print("Prompt format description:", PROMPT_FORMAT_DESCRIPTION)
# print("Query measurement:", selected_query_region_description())
# print("Effective N_TEST_QUERIES_PER_LABEL:", N_TEST_QUERIES_PER_LABEL)
# print("DISCOVERY_N_PER_LABEL:", DISCOVERY_N_PER_LABEL)
# print("HOLDOUT_N_PER_LABEL:", int(N_TEST_QUERIES_PER_LABEL) - int(DISCOVERY_N_PER_LABEL))
# print("Fixed test queries:", expected_queries)
# print("Positions tested:", POSITIONS_TO_TEST)
# print("Total prompts:", len(position_scan_plan_df))
# print("Expected count per (position, varied label, query label):", N_TEST_QUERIES_PER_LABEL)
# print("Fixed context label counts for M-1:", {LABEL_TO_WORD[k]: v for k, v in balanced_label_counts(NUM_DEMOS_TOTAL - 1, LABEL_IDS, seed=SEED).items()})
# print("Split counts in prompts:")
# display(position_scan_plan_df["split"].value_counts().rename_axis("split").reset_index(name="n_prompts"))

# display(position_scan_plan_df.head(12))


In [ ]:
# ============================================================
# Prompt construction and position-scan sampling plan
# ============================================================

import re
import random
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

QUERY_START_SENTINEL = "ZXQ_QUERY_START_4f4cc908"
QUERY_END_SENTINEL = "ZXQ_QUERY_END_4f4cc908"


# ------------------------------------------------------------
# Canonical TREC labels
# ------------------------------------------------------------
# TREC is special because some HF mirrors expose coarse labels
# in a different integer order. We force the canonical order used
# by this notebook:
#   0: Abbreviation
#   1: Entity
#   2: Description
#   3: Human
#   4: Location
#   5: Numeric
# ------------------------------------------------------------

if str(TASK_KEY).lower() == "trec":
    LABEL_TO_WORD = {
        0: "Abbreviation",
        1: "Entity",
        2: "Description",
        3: "Human",
        4: "Location",
        5: "Numeric",
    }
    WORD_TO_LABEL = {v: k for k, v in LABEL_TO_WORD.items()}
    LABEL_IDS = list(LABEL_TO_WORD.keys())
    NUM_LABELS = len(LABEL_IDS)

    # Prefer canonicalized coarse fields.
    LABEL_FIELD_CANDIDATES = [
        "coarse_label",
        "label_coarse",
        "label-coarse",
        "label_coarse_original",
        "label_coarse_text",
        "label",
    ]

    TEXT_FIELD_CANDIDATES = [
        "question",
        "text",
    ]


# ------------------------------------------------------------
# LAST_K helpers
# ------------------------------------------------------------

def normalize_last_k_spec(last_k=LAST_K):
    if last_k is None:
        return None

    if isinstance(last_k, (int, np.integer)):
        k = int(last_k)
        if k <= 0:
            raise ValueError(
                f"Integer LAST_K must be positive, got {last_k}."
            )
        return list(range(-k, 0))

    vals = [int(x) for x in list(last_k)]

    if len(vals) == 0:
        raise ValueError(
            "LAST_K must be None, a positive integer, "
            "or a non-empty list of integer offsets."
        )

    return vals


def measurement_selection_slug(last_k=LAST_K) -> str:
    if last_k is None:
        return "all_query_text_tokens"

    if isinstance(last_k, (int, np.integer)):
        return f"full_prompt_last_{int(last_k)}_tokens"

    vals = [int(x) for x in list(last_k)]

    def _fmt(v):
        return f"m{abs(v)}" if v < 0 else f"p{v}"

    return "full_prompt_offsets_" + "_".join(_fmt(v) for v in vals)


def selected_query_region_description() -> str:
    if LAST_K is None:
        return "all tokens overlapping the final query text span"

    return (
        f"token offsets {normalize_last_k_spec(LAST_K)} from the end of the "
        f"full rendered prompt token sequence "
        f"(MEASURE_LAST_K_FROM_FULL_PROMPT={MEASURE_LAST_K_FROM_FULL_PROMPT}). "
        "With LAST_K=[-1], this selects the terminal chat token, e.g. "
        "<|im_end|>. LAST_K=[-2] selects the token immediately before "
        "<|im_end|>, usually the final label word/token."
    )


# ------------------------------------------------------------
# Generic task text / label helpers
# ------------------------------------------------------------

def _example_keys(example) -> List[str]:
    try:
        return list(example.keys())
    except Exception:
        return []


def _has_key(example, key: str) -> bool:
    try:
        return key in example.keys()
    except Exception:
        try:
            example[key]
            return True
        except Exception:
            return False


def _get_value(example, key: str):
    try:
        return example[key]
    except Exception:
        try:
            return example.get(key)
        except Exception:
            raise KeyError(key)


def _maybe_int(x):
    try:
        return int(x)
    except Exception:
        return None


def _normalize_label_string(x: Any) -> str:
    return str(x).strip().lower().replace("_", " ").replace("-", " ")


TREC_SHORT_TO_CANONICAL_ID = {
    "ABBR": 0,
    "ENTY": 1,
    "DESC": 2,
    "HUM": 3,
    "LOC": 4,
    "NUM": 5,
}

TREC_TEXT_TO_SHORT = {
    "abbreviation": "ABBR",
    "abbreviations": "ABBR",
    "entity": "ENTY",
    "entities": "ENTY",
    "description": "DESC",
    "description and abstract concepts": "DESC",
    "human": "HUM",
    "human beings": "HUM",
    "location": "LOC",
    "locations": "LOC",
    "numeric": "NUM",
    "numeric values": "NUM",
}

# SetFit/TREC-QC coarse-label integer order:
#   DESC=0, ENTY=1, ABBR=2, HUM=3, NUM=4, LOC=5
# Convert to canonical notebook order:
#   ABBR=0, ENTY=1, DESC=2, HUM=3, LOC=4, NUM=5
SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID = {
    0: 2,  # DESC -> Description
    1: 1,  # ENTY -> Entity
    2: 0,  # ABBR -> Abbreviation
    3: 3,  # HUM  -> Human
    4: 5,  # NUM  -> Numeric
    5: 4,  # LOC  -> Location
}


def get_label_id(example: Dict[str, Any]) -> int:
    """Return canonical task label id.

    This function is intentionally defined here so the sampling code does not
    depend on helper definitions from another cell.
    """
    task_key = str(TASK_KEY).lower()

    if task_key == "trec":
        # Best case: previous loader already canonicalized this field.
        if _has_key(example, "coarse_label"):
            raw = _maybe_int(_get_value(example, "coarse_label"))
            if raw is not None and raw in LABEL_TO_WORD:
                return int(raw)

        # SetFit/TREC-QC exposes the original coarse string: ABBR, ENTY, DESC, ...
        if _has_key(example, "label_coarse_original"):
            raw = str(_get_value(example, "label_coarse_original")).strip().upper()
            if raw in TREC_SHORT_TO_CANONICAL_ID:
                return int(TREC_SHORT_TO_CANONICAL_ID[raw])

        # Some mirrors expose human-readable coarse labels.
        if _has_key(example, "label_coarse_text"):
            raw = _normalize_label_string(_get_value(example, "label_coarse_text"))
            if raw in TREC_TEXT_TO_SHORT:
                short = TREC_TEXT_TO_SHORT[raw]
                return int(TREC_SHORT_TO_CANONICAL_ID[short])

        # Original TREC-style field may be named "label-coarse".
        # If this is the original TREC dataset, the integer order is usually
        # already ABBR, ENTY, DESC, HUM, LOC, NUM, i.e. canonical here.
        if _has_key(example, "label-coarse"):
            val = _get_value(example, "label-coarse")

            raw_short = str(val).strip().upper()
            if raw_short in TREC_SHORT_TO_CANONICAL_ID:
                return int(TREC_SHORT_TO_CANONICAL_ID[raw_short])

            raw = _maybe_int(val)
            if raw is not None and raw in LABEL_TO_WORD:
                return int(raw)

        # SetFit/TREC-QC field. Use SetFit-specific remapping unless the
        # dataset was already canonicalized into `coarse_label` above.
        if _has_key(example, "label_coarse"):
            val = _get_value(example, "label_coarse")

            raw_short = str(val).strip().upper()
            if raw_short in TREC_SHORT_TO_CANONICAL_ID:
                return int(TREC_SHORT_TO_CANONICAL_ID[raw_short])

            raw = _maybe_int(val)
            if raw is not None:
                dataset_name = str(globals().get("DATASET_NAME", "")).lower()

                if "setfit/trec-qc" in dataset_name or "trec-qc" in dataset_name:
                    if raw in SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID:
                        return int(SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID[raw])

                if raw in LABEL_TO_WORD:
                    return int(raw)

        # Only use `label` as a final fallback for TREC, because in some TREC
        # mirrors it can mean fine-grained label rather than coarse label.
        if _has_key(example, "label"):
            raw = _maybe_int(_get_value(example, "label"))
            if raw is not None and raw in LABEL_TO_WORD:
                return int(raw)

        raise KeyError(
            "Could not find a usable TREC coarse label. "
            f"Available keys={_example_keys(example)}"
        )

    # Generic path for AGNews / Yahoo / other classification datasets.
    for key in LABEL_FIELD_CANDIDATES:
        if _has_key(example, key):
            raw = _maybe_int(_get_value(example, key))

            if raw is None:
                continue

            if raw in LABEL_TO_WORD:
                return int(raw)

            # Some datasets store labels as 1..K instead of 0..K-1.
            if (raw - 1) in LABEL_TO_WORD:
                return int(raw - 1)

            raise ValueError(
                f"Label value {raw} from field {key!r} is not compatible "
                f"with LABEL_TO_WORD={LABEL_TO_WORD}."
            )

    raise KeyError(
        f"Could not find any label field among {LABEL_FIELD_CANDIDATES}. "
        f"Example keys={_example_keys(example)}"
    )


def clean_task_text(
    text: str,
    max_chars: Optional[int] = MAX_DEMO_CHARS,
) -> str:
    text = str(text).replace("\n", " ").replace("\t", " ").strip()
    text = re.sub(r"\s+", " ", text)

    if max_chars is not None and len(text) > int(max_chars):
        text = text[: int(max_chars)].rstrip() + "..."

    return text


def get_example_text(
    example: Dict[str, Any],
    max_chars: Optional[int] = MAX_DEMO_CHARS,
) -> str:
    # Yahoo Answers benefits from preserving its multi-field structure.
    if str(TASK_KEY).lower() == "yahoo":
        pieces = []

        field_labels = {
            "question_title": "Title",
            "question_content": "Question details",
            "best_answer": "Best answer",
            "text": "Text",
            "title": "Title",
            "content": "Content",
            "answer": "Answer",
        }

        for key in TEXT_FIELD_CANDIDATES:
            if _has_key(example, key):
                val = str(_get_value(example, key)).strip()
                if val and val.lower() != "none":
                    pieces.append(f"{field_labels.get(key, key)}: {val}")

        if not pieces:
            raise KeyError(
                f"Could not construct Yahoo text from keys={_example_keys(example)}"
            )

        return clean_task_text("\n".join(pieces), max_chars=max_chars)

    for key in TEXT_FIELD_CANDIDATES:
        if _has_key(example, key):
            return clean_task_text(
                _get_value(example, key),
                max_chars=max_chars,
            )

    # Extra fallback for TREC mirrors.
    for key in ["question", "text", "sentence"]:
        if _has_key(example, key):
            return clean_task_text(
                _get_value(example, key),
                max_chars=max_chars,
            )

    raise KeyError(
        f"Could not find text field among {TEXT_FIELD_CANDIDATES}. "
        f"Example keys={_example_keys(example)}"
    )


def label_word_for_example(example: Dict[str, Any]) -> str:
    return LABEL_TO_WORD[int(get_label_id(example))]


def format_demo_text_block(example: Dict[str, Any], idx: int) -> str:
    text = get_example_text(example, max_chars=MAX_DEMO_CHARS)
    label = label_word_for_example(example)

    return (
        f"Example {idx}\n"
        f"{INPUT_FIELD_NAME}:\n{text}\n"
        f"{OUTPUT_FIELD_NAME}:\n{label}"
    )


def format_user_request(
    example: Dict[str, Any],
    idx: int,
    *,
    mark_text: bool,
) -> str:
    max_chars = MAX_QUERY_CHARS if mark_text else MAX_DEMO_CHARS
    text = get_example_text(example, max_chars=max_chars)

    if mark_text:
        text = f"{QUERY_START_SENTINEL}{text}{QUERY_END_SENTINEL}"

    return (
        f"Example {idx}\n"
        f"{INPUT_FIELD_NAME}:\n{text}\n"
        f"{OUTPUT_FIELD_NAME}:"
    )


def format_assistant_label(example: Dict[str, Any]) -> str:
    return label_word_for_example(example)


# ------------------------------------------------------------
# Qwen chat rendering
# ------------------------------------------------------------

def manual_qwen_chat_template(messages: List[Dict[str, str]]) -> str:
    """Fallback ChatML renderer matching Qwen-style <|im_start|>/<|im_end|>."""
    parts = []

    for msg in messages:
        role = str(msg["role"])
        content = str(msg["content"])
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>\n")

    return "".join(parts).rstrip()


def apply_qwen_chat_template(messages: List[Dict[str, str]]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    except TypeError:
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        except Exception:
            return manual_qwen_chat_template(messages)
    except Exception:
        return manual_qwen_chat_template(messages)


def render_prompt_from_messages_with_query_span(
    messages: List[Dict[str, str]],
) -> Tuple[str, Tuple[int, int]]:
    rendered = apply_qwen_chat_template(messages)

    s = rendered.find(QUERY_START_SENTINEL)
    e = rendered.find(QUERY_END_SENTINEL)

    if s < 0 or e < 0 or e <= s:
        tail = rendered[:1000] + "\n...\n" + rendered[-1000:]
        raise ValueError(
            "Could not locate query sentinels in rendered prompt. "
            f"Excerpt:\n{tail}"
        )

    query_text_start_marked = s + len(QUERY_START_SENTINEL)
    query_text = rendered[query_text_start_marked:e]

    # Remove sentinels while preserving clean query span.
    prompt = (
        rendered[:s]
        + query_text
        + rendered[e + len(QUERY_END_SENTINEL):]
    )

    query_char_span = (s, s + len(query_text))

    return prompt.rstrip(), query_char_span


# ------------------------------------------------------------
# Prompt-format builders
# ------------------------------------------------------------

def build_prompt_format1_system_demos(
    demos: List[Dict[str, Any]],
    query_example: Dict[str, Any],
) -> Tuple[str, Tuple[int, int]]:
    """Format 1: all demonstrations in system; final query in user; gold label in assistant."""
    demo_blocks = [
        format_demo_text_block(d, i + 1)
        for i, d in enumerate(demos)
    ]

    system_sections = [TASK_INSTRUCTION]

    if len(demo_blocks) > 0:
        system_sections.append(
            "Demonstrations:\n\n" + "\n\n".join(demo_blocks)
        )

    system_content = "\n\n".join(system_sections)

    user_content = format_user_request(
        query_example,
        len(demos) + 1,
        mark_text=True,
    )

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": format_assistant_label(query_example)},
    ]

    return render_prompt_from_messages_with_query_span(messages)


def build_prompt_format2_wrapped_demos(
    demos: List[Dict[str, Any]],
    query_example: Dict[str, Any],
) -> Tuple[str, Tuple[int, int]]:
    """Format 2: each demonstration is user request + assistant label."""
    messages = [
        {
            "role": "system",
            "content": TASK_INSTRUCTION,
        }
    ]

    for i, demo in enumerate(demos, start=1):
        messages.append({
            "role": "user",
            "content": format_user_request(
                demo,
                i,
                mark_text=False,
            ),
        })

        messages.append({
            "role": "assistant",
            "content": format_assistant_label(demo),
        })

    messages.append({
        "role": "user",
        "content": format_user_request(
            query_example,
            len(demos) + 1,
            mark_text=True,
        ),
    })

    messages.append({
        "role": "assistant",
        "content": format_assistant_label(query_example),
    })

    return render_prompt_from_messages_with_query_span(messages)


def build_prompt(
    demos: List[Dict[str, Any]],
    query_example: Dict[str, Any],
) -> Tuple[str, Tuple[int, int]]:
    if PROMPT_FORMAT_NAME == "format1_system_demos_user_query_assistant_label":
        return build_prompt_format1_system_demos(
            demos,
            query_example,
        )

    if PROMPT_FORMAT_NAME == "format2_wrapped_user_assistant_demos":
        return build_prompt_format2_wrapped_demos(
            demos,
            query_example,
        )

    raise ValueError(
        f"Unknown PROMPT_FORMAT_NAME: {PROMPT_FORMAT_NAME}"
    )


# ------------------------------------------------------------
# Position-scan sampling plan
# ------------------------------------------------------------
# No prefix / first-encounter constraint is applied.
# For TREC, labels with fewer than N test examples are sampled with replacement.
# ------------------------------------------------------------

ALLOW_TEST_QUERY_RESAMPLING_WITH_REPLACEMENT = True
KEEP_DISCOVERY_HOLDOUT_SOURCE_EXAMPLES_DISJOINT_WHEN_OVERSAMPLING = True


if POSITIONS_TO_TEST is None:
    POSITIONS_TO_TEST = list(range(1, int(NUM_DEMOS_TOTAL) + 1))
else:
    POSITIONS_TO_TEST = [int(p) for p in POSITIONS_TO_TEST]

for p in POSITIONS_TO_TEST:
    if int(p) < 1 or int(p) > int(NUM_DEMOS_TOTAL):
        raise ValueError(
            f"Invalid varied position {p}. "
            f"Expected positions in 1..NUM_DEMOS_TOTAL={NUM_DEMOS_TOTAL}."
        )


def index_by_label(dataset) -> Dict[int, List[int]]:
    by_label = {
        int(l): []
        for l in LABEL_IDS
    }

    for i, row in enumerate(dataset):
        label_id = int(get_label_id(row))

        if label_id in by_label:
            by_label[label_id].append(int(i))

    return by_label


train_by_label = index_by_label(train_data)
test_by_label = index_by_label(test_data)

print("Train label counts:", {
    LABEL_TO_WORD[k]: len(v)
    for k, v in train_by_label.items()
})

print("Test label counts:", {
    LABEL_TO_WORD[k]: len(v)
    for k, v in test_by_label.items()
})


N_TEST_QUERIES_PER_LABEL = int(N_TEST_QUERIES_PER_LABEL)

if N_TEST_QUERIES_PER_LABEL < 1:
    raise ValueError(
        f"N_TEST_QUERIES_PER_LABEL must be positive, got {N_TEST_QUERIES_PER_LABEL}."
    )

if N_TEST_QUERIES_PER_LABEL > 1:
    DISCOVERY_N_PER_LABEL = int(
        round(float(DISCOVERY_FRACTION) * int(N_TEST_QUERIES_PER_LABEL))
    )

    DISCOVERY_N_PER_LABEL = max(
        1,
        min(
            int(N_TEST_QUERIES_PER_LABEL) - 1,
            int(DISCOVERY_N_PER_LABEL),
        ),
    )
else:
    DISCOVERY_N_PER_LABEL = 1

HOLDOUT_N_PER_LABEL = int(N_TEST_QUERIES_PER_LABEL) - int(DISCOVERY_N_PER_LABEL)


def balanced_label_counts(
    total: int,
    labels: Sequence[int],
    seed: int,
) -> Dict[int, int]:
    """Distribute total examples across labels as uniformly as possible."""
    labels = list(map(int, labels))

    base = int(total) // len(labels)
    remainder = int(total) % len(labels)

    rng = random.Random(int(seed))
    shuffled = labels[:]
    rng.shuffle(shuffled)

    counts = {
        int(l): int(base)
        for l in labels
    }

    for l in shuffled[:remainder]:
        counts[int(l)] += 1

    return counts


def _sample_without_or_with_replacement(
    pool: Sequence[int],
    n: int,
    rng: random.Random,
    *,
    label_id: int,
    split_name: str,
) -> Tuple[List[int], bool]:
    pool = list(map(int, pool))
    n = int(n)

    if n <= 0:
        return [], False

    if len(pool) == 0:
        raise ValueError(
            f"No available test examples for label {label_id} "
            f"({LABEL_TO_WORD[int(label_id)]}) in split {split_name}."
        )

    if len(pool) >= n:
        return rng.sample(pool, n), False

    if not ALLOW_TEST_QUERY_RESAMPLING_WITH_REPLACEMENT:
        raise ValueError(
            f"Not enough {TASK_DISPLAY_NAME} test examples for label "
            f"{label_id} ({LABEL_TO_WORD[int(label_id)]}): "
            f"{len(pool)} < requested {n}. "
            "Set ALLOW_TEST_QUERY_RESAMPLING_WITH_REPLACEMENT=True."
        )

    return rng.choices(pool, k=n), True


def sample_fixed_test_query_indices(
    per_label: int = None,
    seed: int = SEED,
) -> pd.DataFrame:
    """Sample exactly `per_label` test-query draws per label.

    For underrepresented labels, e.g. TREC Abbreviation, this samples with replacement.
    When possible, discovery and holdout are drawn from disjoint source pools.
    """
    if per_label is None:
        per_label = int(N_TEST_QUERIES_PER_LABEL)

    per_label = int(per_label)
    rng = random.Random(int(seed))

    rows = []
    oversampling_rows = []

    for label_id in LABEL_IDS:
        label_id = int(label_id)
        pool = list(map(int, test_by_label[label_id]))

        if len(pool) == 0:
            raise ValueError(
                f"No test examples found for label {label_id} "
                f"({LABEL_TO_WORD[label_id]})."
            )

        if len(pool) >= per_label:
            chosen = rng.sample(pool, per_label)
            sampled_with_replacement = False

            discovery_chosen = chosen[:int(DISCOVERY_N_PER_LABEL)]
            holdout_chosen = chosen[int(DISCOVERY_N_PER_LABEL):]

        else:
            sampled_with_replacement = True

            if (
                KEEP_DISCOVERY_HOLDOUT_SOURCE_EXAMPLES_DISJOINT_WHEN_OVERSAMPLING
                and len(pool) >= 2
                and HOLDOUT_N_PER_LABEL > 0
            ):
                shuffled_pool = pool[:]
                rng.shuffle(shuffled_pool)

                n_discovery_source = int(
                    round(len(shuffled_pool) * float(DISCOVERY_FRACTION))
                )

                n_discovery_source = max(
                    1,
                    min(
                        len(shuffled_pool) - 1,
                        n_discovery_source,
                    ),
                )

                discovery_pool = shuffled_pool[:n_discovery_source]
                holdout_pool = shuffled_pool[n_discovery_source:]

                discovery_chosen, disc_repl = _sample_without_or_with_replacement(
                    discovery_pool,
                    int(DISCOVERY_N_PER_LABEL),
                    rng,
                    label_id=label_id,
                    split_name="discovery",
                )

                holdout_chosen, hold_repl = _sample_without_or_with_replacement(
                    holdout_pool,
                    int(HOLDOUT_N_PER_LABEL),
                    rng,
                    label_id=label_id,
                    split_name="holdout",
                )

                sampled_with_replacement = bool(disc_repl or hold_repl)

            else:
                chosen, sampled_with_replacement = _sample_without_or_with_replacement(
                    pool,
                    per_label,
                    rng,
                    label_id=label_id,
                    split_name="all",
                )

                discovery_chosen = chosen[:int(DISCOVERY_N_PER_LABEL)]
                holdout_chosen = chosen[int(DISCOVERY_N_PER_LABEL):]

            oversampling_rows.append({
                "label_id": label_id,
                "label": LABEL_TO_WORD[label_id],
                "available_test_examples": len(pool),
                "requested_test_query_draws": per_label,
                "sampled_with_replacement": bool(sampled_with_replacement),
                "discovery_draws": len(discovery_chosen),
                "holdout_draws": len(holdout_chosen),
            })

        label_draws = [
            ("discovery", discovery_chosen),
            ("holdout", holdout_chosen),
        ]

        q_pos = 0

        for split_name, chosen_indices in label_draws:
            seen_counts = {}

            for test_index in chosen_indices:
                test_index = int(test_index)
                seen_counts[test_index] = seen_counts.get(test_index, 0) + 1

                rows.append({
                    "query_global_id": len(rows),
                    "query_label_id": label_id,
                    "query_label": LABEL_TO_WORD[label_id],
                    "query_index_within_label": int(q_pos),
                    "test_index": test_index,
                    "split": str(split_name),
                    "test_source_pool_size": int(len(pool)),
                    "test_sampled_with_replacement": bool(sampled_with_replacement),
                    "test_index_occurrence_within_label_split": int(
                        seen_counts[test_index]
                    ),
                })

                q_pos += 1

        if q_pos != per_label:
            raise RuntimeError(
                f"Internal sampling error for label {label_id}: "
                f"sampled {q_pos}, expected {per_label}."
            )

    oversampling_df = pd.DataFrame(oversampling_rows)

    if not oversampling_df.empty:
        print("Labels requiring test-query oversampling:")
        display(oversampling_df)

    return pd.DataFrame(rows)


def sample_fixed_context_pool(query_global_id: int) -> Tuple[int, ...]:
    """Sample the M-1 fixed demonstrations reused across all position scans.

    No prefix-label exclusion constraint is applied. For a position-k DPS search,
    labels in positions 1...(k-1) are sampled normally and may include the test-query label.
    """
    total_fixed = int(NUM_DEMOS_TOTAL) - 1

    counts = balanced_label_counts(
        total_fixed,
        LABEL_IDS,
        seed=SEED + 17_000 + 101 * int(query_global_id),
    )

    rng = random.Random(
        SEED
        + 31_000
        + 1_003 * int(query_global_id)
    )

    fixed_indices = []

    for label_id in LABEL_IDS:
        label_id = int(label_id)
        c = int(counts[label_id])

        if c <= 0:
            continue

        pool = list(map(int, train_by_label[label_id]))

        if len(pool) == 0:
            raise ValueError(
                f"No train demos available for label {label_id} "
                f"({LABEL_TO_WORD[label_id]})."
            )

        if len(pool) >= c:
            fixed_indices.extend(rng.sample(pool, c))
        else:
            print(
                f"Warning: only {len(pool)} train demos for label {label_id} "
                f"({LABEL_TO_WORD[label_id]}), but need {c}; "
                "sampling train demos with replacement."
            )
            fixed_indices.extend(rng.choices(pool, k=c))

    rng.shuffle(fixed_indices)

    assert len(fixed_indices) == total_fixed

    return tuple(map(int, fixed_indices))


def sample_variable_demo_indices_by_label(
    query_global_id: int,
    fixed_context_indices: Sequence[int],
) -> Dict[int, int]:
    """Sample one variable demonstration per label, reused across all varied positions."""
    exclude = set(map(int, fixed_context_indices))
    out = {}

    for label_id in LABEL_IDS:
        label_id = int(label_id)

        pool = [
            int(i)
            for i in train_by_label[label_id]
            if int(i) not in exclude
        ]

        if len(pool) == 0:
            # Defensive fallback for tiny datasets.
            pool = list(map(int, train_by_label[label_id]))

        if len(pool) == 0:
            raise ValueError(
                f"No available variable demos for label {label_id} "
                f"({LABEL_TO_WORD[label_id]})."
            )

        rng = random.Random(
            SEED
            + 59_000
            + 10_007 * int(query_global_id)
            + int(label_id)
        )

        out[label_id] = int(rng.choice(pool))

    return out


def insert_variable_at_position(
    fixed_context_indices: Sequence[int],
    variable_idx: int,
    varied_position: int,
) -> Tuple[int, ...]:
    """Insert variable_idx at 1-indexed varied_position into the fixed M-1 context pool."""
    fixed = list(map(int, fixed_context_indices))

    p0 = int(varied_position) - 1

    if p0 < 0 or p0 > len(fixed):
        raise IndexError(
            f"varied_position must be in 1..{len(fixed) + 1}, "
            f"got {varied_position}"
        )

    return tuple(
        fixed[:p0]
        + [int(variable_idx)]
        + fixed[p0:]
    )


def build_position_scan_plan() -> pd.DataFrame:
    query_df = sample_fixed_test_query_indices()

    rows = []

    for r in query_df.itertuples(index=False):
        fixed_context_indices = sample_fixed_context_pool(
            query_global_id=int(r.query_global_id),
        )

        variable_by_label = sample_variable_demo_indices_by_label(
            query_global_id=int(r.query_global_id),
            fixed_context_indices=fixed_context_indices,
        )

        for varied_position in POSITIONS_TO_TEST:
            p0 = int(varied_position) - 1

            for varied_label_id in LABEL_IDS:
                varied_label_id = int(varied_label_id)

                variable_idx = int(variable_by_label[varied_label_id])

                demo_indices = insert_variable_at_position(
                    fixed_context_indices=fixed_context_indices,
                    variable_idx=variable_idx,
                    varied_position=int(varied_position),
                )

                non_varied_demo_indices = tuple(
                    demo_indices[:p0]
                    + demo_indices[p0 + 1:]
                )

                rows.append({
                    "query_global_id": int(r.query_global_id),
                    "query_label_id": int(r.query_label_id),
                    "query_label": str(r.query_label),
                    "query_index_within_label": int(r.query_index_within_label),
                    "test_index": int(r.test_index),
                    "split": str(r.split),
                    "test_source_pool_size": int(r.test_source_pool_size),
                    "test_sampled_with_replacement": bool(
                        r.test_sampled_with_replacement
                    ),
                    "test_index_occurrence_within_label_split": int(
                        r.test_index_occurrence_within_label_split
                    ),
                    "varied_position": int(varied_position),
                    "varied_label_id": int(varied_label_id),
                    "varied_label": LABEL_TO_WORD[varied_label_id],
                    "variable_demo_index": int(variable_idx),
                    "fixed_context_indices": tuple(fixed_context_indices),
                    "non_varied_demo_indices": tuple(non_varied_demo_indices),
                    "demo_indices": tuple(demo_indices),
                })

    return pd.DataFrame(rows)


position_scan_plan_df = build_position_scan_plan()


# ------------------------------------------------------------
# Structural sanity checks
# ------------------------------------------------------------

expected_queries = int(NUM_LABELS) * int(N_TEST_QUERIES_PER_LABEL)

expected_prompts = (
    expected_queries
    * len(POSITIONS_TO_TEST)
    * int(NUM_LABELS)
)

assert len(position_scan_plan_df) == expected_prompts, (
    len(position_scan_plan_df),
    expected_prompts,
)

assert position_scan_plan_df["query_global_id"].nunique() == expected_queries

assert (
    position_scan_plan_df
    .groupby(["query_global_id", "varied_position"])
    .size()
    .eq(NUM_LABELS)
    .all()
)

assert (
    position_scan_plan_df
    .groupby(["query_global_id", "varied_position"])["non_varied_demo_indices"]
    .nunique()
    .eq(1)
    .all()
)

assert (
    position_scan_plan_df
    .groupby(["query_global_id", "varied_label_id"])["variable_demo_index"]
    .nunique()
    .eq(1)
    .all()
)


print("Task:", TASK_DISPLAY_NAME)
print("Dataset:", DATASET_NAME)
print("Prompt format:", PROMPT_FORMAT_NAME)
print("Prompt format description:", PROMPT_FORMAT_DESCRIPTION)
print("Query measurement:", selected_query_region_description())
print("Requested/effective N_TEST_QUERIES_PER_LABEL:", N_TEST_QUERIES_PER_LABEL)
print("DISCOVERY_N_PER_LABEL:", DISCOVERY_N_PER_LABEL)
print("HOLDOUT_N_PER_LABEL:", HOLDOUT_N_PER_LABEL)
print("Fixed test-query draws:", expected_queries)
print("Positions tested:", POSITIONS_TO_TEST)
print("Total prompts:", len(position_scan_plan_df))

print(
    "Expected total prompts formula:",
    f"{NUM_LABELS} labels × {N_TEST_QUERIES_PER_LABEL} queries/label "
    f"× {len(POSITIONS_TO_TEST)} positions × {NUM_LABELS} varied labels "
    f"= {expected_prompts}",
)

print(
    "Expected count per (position, varied label, query label):",
    N_TEST_QUERIES_PER_LABEL,
)

print(
    "Fixed context label counts for M-1:",
    {
        LABEL_TO_WORD[k]: v
        for k, v in balanced_label_counts(
            int(NUM_DEMOS_TOTAL) - 1,
            LABEL_IDS,
            seed=SEED,
        ).items()
    },
)

print("Split counts in prompts:")
display(
    position_scan_plan_df["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="n_prompts")
)

print("Prompt counts by query label:")
display(
    position_scan_plan_df
    .groupby(["query_label_id", "query_label"])
    .size()
    .reset_index(name="n_prompts")
)

print("Unique test examples actually used per query label:")
display(
    position_scan_plan_df[
        [
            "query_label_id",
            "query_label",
            "query_global_id",
            "test_index",
        ]
    ]
    .drop_duplicates()
    .groupby(["query_label_id", "query_label"])
    .agg(
        query_draws=("query_global_id", "count"),
        unique_test_indices=("test_index", "nunique"),
    )
    .reset_index()
)

display(position_scan_plan_df.head(12))

## 5. Sanity check: one rendered prompt and selected tokens


In [ ]:
# ============================================================
# Sanity check: print one rendered prompt and selected tokens
# ============================================================

SANITY_QUERY_GLOBAL_ID = 0
SANITY_VARIED_POSITION = 1
SANITY_VARIED_LABEL_ID = 0

sanity_row = position_scan_plan_df[
    (position_scan_plan_df["query_global_id"] == int(SANITY_QUERY_GLOBAL_ID))
    & (position_scan_plan_df["varied_position"] == int(SANITY_VARIED_POSITION))
    & (position_scan_plan_df["varied_label_id"] == int(SANITY_VARIED_LABEL_ID))
].iloc[0]


def build_prompt_from_position_plan_row(row) -> Tuple[str, Tuple[int, int]]:
    demo_indices = tuple(map(int, row["demo_indices"])) if isinstance(row, pd.Series) else tuple(map(int, row.demo_indices))
    test_index = int(row["test_index"] if isinstance(row, pd.Series) else row.test_index)
    demos = [train_data[int(i)] for i in demo_indices]
    query_example = test_data[test_index]
    return build_prompt(demos, query_example)


sanity_prompt, sanity_query_span = build_prompt_from_position_plan_row(sanity_row)

print("=" * 100)
print("SANITY PROMPT")
print("query_global_id:", int(sanity_row["query_global_id"]))
print("split:", sanity_row["split"])
print("test query label:", sanity_row["query_label"])
print("varied position:", int(sanity_row["varied_position"]))
print("varied label:", sanity_row["varied_label"])
print("query char span:", sanity_query_span)
print("query span text:")
print(repr(sanity_prompt[sanity_query_span[0]:sanity_query_span[1]]))
print("-" * 100)
print(sanity_prompt[:DISPLAY_PROMPT_SNIPPET_CHARS])
if len(sanity_prompt) > DISPLAY_PROMPT_SNIPPET_CHARS:
    print("\n... [prompt truncated for display] ...")
print("=" * 100)


In [ ]:
# ============================================================
# Token-selection utilities and selected-token sanity check
# ============================================================


def char_span_to_token_indices(offset_mapping: List[Tuple[int, int]], char_span: Tuple[int, int]) -> List[int]:
    cs, ce = int(char_span[0]), int(char_span[1])
    out = []
    for i, (s, e) in enumerate(offset_mapping):
        # Fast tokenizers often use (0, 0) for synthetic special tokens. We still
        # include such tokens when LAST_K is measured from the full prompt, but
        # not when converting a character span to text-overlapping tokens.
        if s == 0 and e == 0:
            continue
        if e > cs and s < ce:
            out.append(int(i))
    if not out:
        raise ValueError(f"No tokens found for char span {char_span}")
    return out


def select_token_indices_from_sequence(n_tokens: int, last_k=LAST_K) -> List[int]:
    offsets = normalize_last_k_spec(last_k)
    if offsets is None:
        raise ValueError("select_token_indices_from_sequence is only used when LAST_K is not None.")

    selected = []
    n = int(n_tokens)
    for off in offsets:
        pos = n + int(off) if int(off) < 0 else int(off)
        if pos < 0 or pos >= n:
            raise IndexError(
                f"LAST_K offset {off} is out of bounds for a prompt with {n} tokens. "
                f"Valid negative offsets are [-{n}, ..., -1]."
            )
        if pos not in selected:
            selected.append(int(pos))
    return selected


def get_query_measurement_token_indices(
    input_ids: Sequence[int],
    offset_mapping: List[Tuple[int, int]],
    query_char_span: Tuple[int, int],
    last_k=LAST_K,
) -> Tuple[List[int], List[int]]:
    """Return query-article span tokens and the actually selected measurement tokens.

    Important behavior requested here:
    - LAST_K is None: select all tokens overlapping the final query article span.
    - LAST_K is not None: select offsets from the full rendered prompt token sequence.
      Thus LAST_K=[-1] selects the terminal token of the prompt, e.g. the final
      <|im_end|> after the gold assistant label, not the label word itself.
    """
    span_token_ids = char_span_to_token_indices(offset_mapping, query_char_span)

    offsets = normalize_last_k_spec(last_k)
    if offsets is None:
        selected_token_ids = list(span_token_ids)
    else:
        if not MEASURE_LAST_K_FROM_FULL_PROMPT:
            raise ValueError(
                "This notebook variant expects MEASURE_LAST_K_FROM_FULL_PROMPT=True "
                "whenever LAST_K is not None."
            )
        selected_token_ids = select_token_indices_from_sequence(len(input_ids), last_k=last_k)

    return span_token_ids, selected_token_ids


def describe_query_token_selection(
    prompt: str,
    query_char_span: Tuple[int, int],
    last_k=LAST_K,
    max_rows: Optional[int] = None,
) -> pd.DataFrame:
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )
    input_ids = enc["input_ids"][0].detach().cpu().tolist()
    offset_mapping = enc["offset_mapping"][0].tolist()
    span_token_ids, selected_token_ids = get_query_measurement_token_indices(
        input_ids,
        offset_mapping,
        query_char_span,
        last_k=last_k,
    )

    query_span_set = set(span_token_ids)
    selected_set = set(selected_token_ids)
    span_pos_map = {tok_idx: local_pos for local_pos, tok_idx in enumerate(span_token_ids)}

    rows = []
    for tok_idx in range(len(input_ids)):
        s0, e0 = offset_mapping[tok_idx]
        local_pos = span_pos_map.get(tok_idx, None)
        rows.append({
            "absolute_token_idx": int(tok_idx),
            "offset_from_end_full_prompt": int(tok_idx) - len(input_ids),
            "local_pos_in_query_article_span": None if local_pos is None else int(local_pos),
            "offset_from_end_query_article_span": None if local_pos is None else int(local_pos) - len(span_token_ids),
            "in_query_article_span": bool(tok_idx in query_span_set),
            "selected_for_activation": bool(tok_idx in selected_set),
            "token_id": int(input_ids[tok_idx]),
            "decoded_token": repr(tokenizer.decode([int(input_ids[tok_idx])])),
            "char_start": int(s0),
            "char_end": int(e0),
            "text_piece": repr(prompt[s0:e0]) if e0 > s0 else "''",
        })

    df = pd.DataFrame(rows)

    if max_rows is not None and len(df) > max_rows:
        keep = np.zeros(len(df), dtype=bool)
        selected_positions = df.index[df["selected_for_activation"]].tolist()
        query_positions = df.index[df["in_query_article_span"]].tolist()

        # Always keep windows around the actually selected tokens.
        for pos in selected_positions:
            keep[max(0, pos - 12): min(len(df), pos + 13)] = True

        # Also keep the start/end of the query article span for sanity checking.
        for pos in (query_positions[:1] + query_positions[-1:]):
            keep[max(0, pos - 8): min(len(df), pos + 9)] = True

        selected_window_df = df.loc[keep].copy()
        if len(selected_window_df) > max_rows:
            priority = selected_window_df["selected_for_activation"].astype(int) * 10
            priority += selected_window_df["in_query_article_span"].astype(int)
            selected_window_df = (
                selected_window_df
                .assign(_priority=priority)
                .sort_values(["_priority", "absolute_token_idx"], ascending=[False, True])
                .head(int(max_rows))
                .sort_values("absolute_token_idx")
                .drop(columns=["_priority"])
            )
        df = selected_window_df.reset_index(drop=True)

    return df


print("Selected tokens used for SAE pre-activation averaging")
print("LAST_K:", LAST_K)
print("Measurement:", selected_query_region_description())
selection_df = describe_query_token_selection(sanity_prompt, sanity_query_span, last_k=LAST_K, max_rows=120)
display(selection_df)

selected_rows = selection_df[selection_df["selected_for_activation"]]
if len(selected_rows) > 0:
    print("Selected token(s):")
    display(selected_rows)

print("Explicit LAST_K offset sanity check for wrapped prompt format:")
for check_last_k in ([-1], [-2]):
    check_df = describe_query_token_selection(
        sanity_prompt,
        sanity_query_span,
        last_k=check_last_k,
        max_rows=80,
    )
    print(f"LAST_K={check_last_k}")
    display(check_df[check_df["selected_for_activation"]])

print("Label tokenization sanity check:")
label_token_rows = []
for label_id, label_word in LABEL_TO_WORD.items():
    ids = tokenizer(label_word, add_special_tokens=False)["input_ids"]
    label_token_rows.append({
        "label_id": int(label_id),
        "label": label_word,
        "token_ids": ids,
        "decoded_tokens": [repr(tokenizer.decode([int(t)])) for t in ids],
        "n_tokens": len(ids),
    })
display(pd.DataFrame(label_token_rows))


## 6. Activation extraction and streaming aggregation

The resulting tensor has shape:

```python
mean_A_by_split[split][position, varied_label, query_label, feature]
```

For example, `mean_A_by_split["holdout"][0, 2, 2, f]` is the held-out mean pre-activation of feature `f` when the varied demo is in the first tested position and both the varied demo and test query have label `Business`.


In [ ]:
# ============================================================
# Activation extraction and streaming aggregation
# ============================================================


def get_layer_output(input_ids: torch.Tensor, attention_mask: torch.Tensor, layer_idx: int) -> torch.Tensor:
    captured = {}
    block = model.get_submodule(f"model.layers.{layer_idx}")

    def hook_fn(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        captured["x"] = x.detach()
        return output

    handle = block.register_forward_hook(hook_fn)
    try:
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    finally:
        handle.remove()
    if "x" not in captured:
        raise RuntimeError(f"Failed to capture layer output at layer {layer_idx}")
    return captured["x"]


@torch.no_grad()
def extract_query_sae_vector(prompt: str, query_char_span: Tuple[int, int]) -> np.ndarray:
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )
    offset_mapping = enc.pop("offset_mapping")[0].tolist()
    input_ids_cpu = enc["input_ids"][0].detach().cpu().tolist()
    _, token_ids = get_query_measurement_token_indices(
        input_ids_cpu,
        offset_mapping,
        query_char_span,
        last_k=LAST_K,
    )

    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    layer_out = get_layer_output(input_ids, attention_mask, TARGET_LAYER)  # [1, seq, d_model]
    idx = torch.tensor(token_ids, device=layer_out.device, dtype=torch.long)
    query_hidden = layer_out[0].index_select(0, idx)  # [selected_tokens, d_model]

    pre = sae.encode_pre(query_hidden.to(sae.device))
    vals = pre if USE_SAE_PREACTIVATIONS else sae.activation_fn(pre)
    vec = vals.mean(dim=0).detach().float().cpu().numpy().astype(np.float32)
    return vec


def position_plan_fingerprint(plan: pd.DataFrame) -> str:
    payload = {
        "seed": SEED,
        "task_key": TASK_KEY,
        "task_display_name": TASK_DISPLAY_NAME,
        "dataset_name": DATASET_NAME,
        "n_test_queries_per_label": int(N_TEST_QUERIES_PER_LABEL),
        "num_demos_total": int(NUM_DEMOS_TOTAL),
        "positions_to_test": [int(p) for p in POSITIONS_TO_TEST],
        "discovery_n_per_label": int(DISCOVERY_N_PER_LABEL),
        "last_k": None if LAST_K is None else [int(x) for x in normalize_last_k_spec(LAST_K)],
        "prompt_format_name": PROMPT_FORMAT_NAME,
        "measurement_rule": "LAST_K offsets are measured from the full rendered prompt when LAST_K is not None",
        "measure_last_k_from_full_prompt": bool(MEASURE_LAST_K_FROM_FULL_PROMPT),
        "target_layer": int(TARGET_LAYER),
        "model_name": MODEL_NAME,
        "sae_repo_id": SAE_REPO_ID,
        "sae_filename": SAE_FILENAME,
        "rows": [
            {
                "query_global_id": int(r.query_global_id),
                "test_index": int(r.test_index),
                "query_label_id": int(r.query_label_id),
                "split": str(r.split),
                "varied_position": int(r.varied_position),
                "varied_label_id": int(r.varied_label_id),
                "demo_indices": tuple(map(int, r.demo_indices)),
            }
            for r in plan.itertuples(index=False)
        ],
    }
    js = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(js.encode("utf-8")).hexdigest()[:16]


RUN_FINGERPRINT = position_plan_fingerprint(position_scan_plan_df)
CACHE_PATH = CACHE_DIR / (
    f"position_scan_{PROMPT_FORMAT_NAME}_{MODEL_SHORT_NAME}_layer{TARGET_LAYER}_"
    f"M{NUM_DEMOS_TOTAL}_Q{N_TEST_QUERIES_PER_LABEL}_P{'-'.join(map(str, POSITIONS_TO_TEST))}_"
    f"{measurement_selection_slug(LAST_K)}_seed{SEED}_{RUN_FINGERPRINT}.npz"
)
print("CACHE_PATH:", CACHE_PATH)


def _init_aggregate_arrays(n_features: int):
    shape = (len(POSITIONS_TO_TEST), NUM_LABELS, NUM_LABELS, int(n_features))
    count_shape = (len(POSITIONS_TO_TEST), NUM_LABELS, NUM_LABELS)
    return {
        "sum": np.zeros(shape, dtype=np.float64),
        "sum_sq": np.zeros(shape, dtype=np.float64),
        "count": np.zeros(count_shape, dtype=np.int64),
    }


def _add_to_aggregate(agg, p_idx: int, c: int, l: int, vec64: np.ndarray) -> None:
    agg["sum"][p_idx, c, l, :] += vec64
    agg["sum_sq"][p_idx, c, l, :] += vec64 * vec64
    agg["count"][p_idx, c, l] += 1


def _finalize_aggregate(agg):
    count = agg["count"]
    denom = np.maximum(count[:, :, :, None], 1)
    mean_A = agg["sum"] / denom
    second_moment = agg["sum_sq"] / denom
    variance_A = np.maximum(second_moment - mean_A * mean_A, 0.0)
    sem_A = np.sqrt(variance_A / denom)
    return mean_A.astype(np.float32), sem_A.astype(np.float32), count


def run_position_scan_activation_experiment(plan: pd.DataFrame):
    n_features = int(sae.n_features)
    split_names = ["all", "discovery", "holdout"]
    aggs = {name: _init_aggregate_arrays(n_features) for name in split_names}

    total = len(plan)
    for done, row in enumerate(plan.itertuples(index=False), start=1):
        demos = [train_data[int(i)] for i in tuple(map(int, row.demo_indices))]
        query_example = test_data[int(row.test_index)]
        prompt, query_span = build_prompt(demos, query_example)
        vec = extract_query_sae_vector(prompt, query_span)
        vec64 = vec.astype(np.float64, copy=False)

        p_idx = POSITION_TO_IDX[int(row.varied_position)]
        c = int(row.varied_label_id)
        l = int(row.query_label_id)

        _add_to_aggregate(aggs["all"], p_idx, c, l, vec64)
        _add_to_aggregate(aggs[str(row.split)], p_idx, c, l, vec64)

        if done % int(PRINT_EVERY) == 0 or done == total:
            print(f"processed {done}/{total} prompts")

    mean_A_by_split, sem_A_by_split, count_A_by_split = {}, {}, {}
    for name in split_names:
        mean_A_by_split[name], sem_A_by_split[name], count_A_by_split[name] = _finalize_aggregate(aggs[name])
    return mean_A_by_split, sem_A_by_split, count_A_by_split


run_meta = {
    "model_name": MODEL_NAME,
    "model_short_name": MODEL_SHORT_NAME,
    "target_layer": TARGET_LAYER,
    "sae_repo_id": SAE_REPO_ID,
    "sae_filename": SAE_FILENAME,
    "task_key": TASK_KEY,
    "task_display_name": TASK_DISPLAY_NAME,
    "dataset_name": DATASET_NAME,
    "n_test_queries_per_label": N_TEST_QUERIES_PER_LABEL,
    "num_demos_total": NUM_DEMOS_TOTAL,
    "positions_to_test": POSITIONS_TO_TEST,
    "discovery_n_per_label": DISCOVERY_N_PER_LABEL,
    "labels": LABEL_TO_WORD,
    "last_k": None if LAST_K is None else [int(x) for x in normalize_last_k_spec(LAST_K)],
    "prompt_format_name": PROMPT_FORMAT_NAME,
    "prompt_format_description": PROMPT_FORMAT_DESCRIPTION,
    "measurement_rule": "LAST_K offsets are measured from the full rendered prompt when LAST_K is not None",
    "measure_last_k_from_full_prompt": MEASURE_LAST_K_FROM_FULL_PROMPT,
    "use_sae_preactivations": USE_SAE_PREACTIVATIONS,
    "prompt_format": PROMPT_FORMAT_DESCRIPTION,
    "run_fingerprint": RUN_FINGERPRINT,
}

split_names = ["all", "discovery", "holdout"]

if USE_CACHE and CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    print("Loading cached position-scan activation aggregates...")
    cached = np.load(CACHE_PATH, allow_pickle=False)
    mean_A_by_split = {name: cached[f"mean_A_{name}"] for name in split_names}
    sem_A_by_split = {name: cached[f"sem_A_{name}"] for name in split_names}
    count_A_by_split = {name: cached[f"count_A_{name}"] for name in split_names}
    cached_meta = json.loads(cached["run_meta_json"].item())
    print("Loaded cache meta:", cached_meta)
else:
    print("Computing position-scan activation aggregates from scratch...")
    mean_A_by_split, sem_A_by_split, count_A_by_split = run_position_scan_activation_experiment(position_scan_plan_df)
    if USE_CACHE:
        print("Saving cache:", CACHE_PATH)
        save_payload = {"run_meta_json": json.dumps(run_meta, sort_keys=True)}
        for name in split_names:
            save_payload[f"mean_A_{name}"] = mean_A_by_split[name]
            save_payload[f"sem_A_{name}"] = sem_A_by_split[name]
            save_payload[f"count_A_{name}"] = count_A_by_split[name]
        np.savez_compressed(CACHE_PATH, **save_payload)

print("Tensor shapes:")
for name in split_names:
    print(name, "mean", mean_A_by_split[name].shape, "sem", sem_A_by_split[name].shape, "count", count_A_by_split[name].shape)

print("Holdout counts per (position, varied label, query label):")
for p_idx, position in IDX_TO_POSITION.items():
    print("position", position)
    display(pd.DataFrame(
        count_A_by_split["holdout"][p_idx],
        index=[LABEL_TO_WORD[i] for i in LABEL_IDS],
        columns=[LABEL_TO_WORD[i] for i in LABEL_IDS],
    ))


## 7. Compute position-wise DPS-A scores and select features on discovery queries

For each varied position `r`, query label `ell`, and feature `f`, define the DPS-A matched-vs-best-alternative gap:

\[
\Delta^{(r)}_{\mathrm{DPS-A}}(\ell, f) = A_f^{(r)}(\ell \rightarrow \ell) - \max_{c \ne \ell} A_f^{(r)}(c \rightarrow \ell).
\]

Positive values mean the matched condition wins for that query label. We select DPS-A candidates using only `r=1` on discovery queries, then evaluate all positions on holdout queries.


In [ ]:
# ============================================================
# Position-wise DPS-A scores and discovery feature selection
# ============================================================

DPS_TIE_TOL = 1e-9


def compute_dps_a_position_stats(mean_A_pos: np.ndarray) -> Dict[str, np.ndarray]:
    '''Compute DPS-A conditions for mean_A_pos[position, varied_label, query_label, feature].'''
    n_positions = int(mean_A_pos.shape[0])
    n_features = int(mean_A_pos.shape[-1])

    condition = np.zeros((n_positions, NUM_LABELS, n_features), dtype=bool)
    gap = np.zeros((n_positions, NUM_LABELS, n_features), dtype=np.float32)

    for p_idx in range(n_positions):
        for l in LABEL_IDS:
            vals = mean_A_pos[p_idx, :, int(l), :]       # [varied_label, feature]
            matched = vals[int(l), :]                   # [feature]
            other_labels = [int(k) for k in LABEL_IDS if int(k) != int(l)]
            other_max = vals[other_labels, :].max(axis=0)

            gap[p_idx, int(l), :] = matched - other_max
            condition[p_idx, int(l), :] = matched >= (vals.max(axis=0) - DPS_TIE_TOL)

    count = condition.sum(axis=1)                       # [position, feature]
    mean_gap = gap.mean(axis=1)                         # [position, feature]
    min_gap = gap.min(axis=1)                           # [position, feature]
    return {
        "condition": condition,
        "gap": gap,
        "count": count,
        "mean_gap": mean_gap,
        "min_gap": min_gap,
    }


stats_by_split = {name: compute_dps_a_position_stats(mean_A_by_split[name]) for name in split_names}
n_features = int(mean_A_by_split["all"].shape[-1])
pos1_idx = POSITION_TO_IDX[1] if 1 in POSITION_TO_IDX else 0
EVAL_SPLIT = "holdout" if int(count_A_by_split["holdout"].sum()) > 0 else "all"
print("Evaluation split:", EVAL_SPLIT)


def build_position_feature_count_df(split_name: str) -> pd.DataFrame:
    stats = stats_by_split[split_name]
    rows = []
    for p_idx, position in IDX_TO_POSITION.items():
        counts = stats["count"][p_idx, :]
        rows.append({
            "split": split_name,
            "varied_position": int(position),
            "num_all4_dps_a_features": int((counts == NUM_LABELS).sum()),
            "num_3plus_dps_a_features": int((counts >= max(NUM_LABELS - 1, 0)).sum()),
            "mean_dps_a_count_all_features": float(counts.mean()),
        })
    return pd.DataFrame(rows)


position_feature_count_df = pd.concat(
    [build_position_feature_count_df(name) for name in split_names],
    ignore_index=True,
)
print("Position-wise DPS-A-like feature counts across all SAE features:")
display(position_feature_count_df)

# Select candidate DPS-A features using only discovery queries and only the first varied position.
discovery_stats = stats_by_split["discovery"]
discovery_global_abs_mean = np.abs(mean_A_by_split["discovery"].mean(axis=(0, 1, 2)))

feature_discovery_df = pd.DataFrame({
    "feature_idx": np.arange(n_features, dtype=int),
    "discovery_pos1_dps_a_count": discovery_stats["count"][pos1_idx, :].astype(int),
    "discovery_pos1_mean_gap": discovery_stats["mean_gap"][pos1_idx, :],
    "discovery_pos1_min_gap": discovery_stats["min_gap"][pos1_idx, :],
    "abs_global_mean_preactivation_discovery": discovery_global_abs_mean,
})
feature_discovery_df["is_discovery_pos1_all4_dps_a"] = feature_discovery_df["discovery_pos1_dps_a_count"].eq(NUM_LABELS)

feature_discovery_df = feature_discovery_df.sort_values(
    ["discovery_pos1_dps_a_count", "discovery_pos1_mean_gap", "discovery_pos1_min_gap"],
    ascending=[False, False, False],
).reset_index(drop=True)

all4_discovery_features = feature_discovery_df[
    feature_discovery_df["is_discovery_pos1_all4_dps_a"]
]["feature_idx"].astype(int).tolist()

if len(all4_discovery_features) > 0:
    selected_dps_a_features = all4_discovery_features
    selection_rule = "all discovery position-1 all-label DPS-A features"
    if MAX_DPS_A_FEATURES_FOR_EVAL is not None and len(selected_dps_a_features) > int(MAX_DPS_A_FEATURES_FOR_EVAL):
        selected_dps_a_features = selected_dps_a_features[: int(MAX_DPS_A_FEATURES_FOR_EVAL)]
        selection_rule += f"; capped to top {MAX_DPS_A_FEATURES_FOR_EVAL} by discovery gap"
else:
    selected_dps_a_features = feature_discovery_df.head(int(TOP_K_FALLBACK_FEATURES))["feature_idx"].astype(int).tolist()
    selection_rule = f"fallback: top {TOP_K_FALLBACK_FEATURES} features by discovery position-1 DPS-A count/gap"

selected_dps_a_features = list(map(int, selected_dps_a_features))
selected_set = set(selected_dps_a_features)
if len(selected_dps_a_features) == 0:
    raise RuntimeError("No selected DPS-A features. Lower thresholds or inspect feature_discovery_df.")

print("Selection rule:", selection_rule)
print("Number of selected DPS-A features:", len(selected_dps_a_features))
print("First selected features:", selected_dps_a_features[:50])
print("Top discovery features:")
display(feature_discovery_df.head(25))


## Filled ToDo 1: top-100 DPS-A features at a specified position

This cell ranks and plots the top DPS-A-like features at a configurable zero-based position index. `TOP_DPS_A_POSITION_INDEX = 0` corresponds to the original first-demo position when `POSITIONS_TO_TEST[0] == 1`. The displayed table includes `average_matched_preactivation`, defined as the average diagonal value `A_f(l -> l)` across labels.


In [ ]:
# ============================================================
# ToDo 1 filled: Top-k DPS-A features at a specified position
# ============================================================

# Position is specified as a ZERO-BASED index into POSITIONS_TO_TEST.
# Example: TOP_DPS_A_POSITION_INDEX = 0 corresponds to the original first-demo scan
# when POSITIONS_TO_TEST[0] == 1.
TOP_DPS_A_POSITION_INDEX = 0
TOP_DPS_A_SPLIT = EVAL_SPLIT          # Options: "discovery", "holdout", or "all".
TOP_DPS_A_N = 100
TOP_DPS_A_REQUIRE_ALL4 = True         # True = keep only features satisfying all task-specific DPS-A label conditions.

# Local plotting defaults. This cell appears before the main visualization-helper cell,
# so define these here instead of relying on later cells.
PLOT_FONT = globals().get("PLOT_FONT", "Times New Roman, Times, serif")


def _axis_title_for_todo(text: str, size: int = 18):
    return dict(
        text=text,
        font=dict(
            size=size,
            family=PLOT_FONT,
        ),
    )


def _safe_label_col_name(label_name: str) -> str:
    return (
        str(label_name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def resolve_position_index(position_index: int) -> int:
    """Resolve a zero-based position index into mean_A_by_split/stats_by_split arrays."""
    position_index = int(position_index)
    if position_index < 0 or position_index >= len(POSITIONS_TO_TEST):
        raise ValueError(
            f"position_index={position_index} is out of range. "
            f"Valid zero-based indices are 0..{len(POSITIONS_TO_TEST) - 1}. "
            f"POSITIONS_TO_TEST={POSITIONS_TO_TEST}"
        )
    return position_index


def build_top_dps_a_features_table(
    position_index: int = TOP_DPS_A_POSITION_INDEX,
    split_name: str = TOP_DPS_A_SPLIT,
    top_n: int = TOP_DPS_A_N,
    require_all4: bool = TOP_DPS_A_REQUIRE_ALL4,
    require_positive_matched: bool = False,
) -> pd.DataFrame:
    """Build a ranked table of DPS-A-like features for a chosen varied-position index.

    Ranking priority:
      1. number of satisfied DPS-A label conditions,
      2. mean matched-vs-best-nonmatched gap,
      3. minimum label-wise gap,
      4. average matched preactivation.

    The average matched preactivation is:
        mean_l A_f(position, varied_label=l, query_label=l).
    """
    if split_name not in stats_by_split:
        raise ValueError(f"Unknown split_name={split_name!r}. Available splits: {list(stats_by_split)}")

    p_idx = resolve_position_index(position_index)
    varied_position = int(IDX_TO_POSITION[p_idx])

    stats = stats_by_split[split_name]
    mean_A = mean_A_by_split[split_name]

    # matched_by_label[label, feature] = A_f(c=l -> l) at the chosen position.
    matched_by_label = np.stack(
        [mean_A[p_idx, int(label_id), int(label_id), :] for label_id in LABEL_IDS],
        axis=0,
    )
    average_matched = matched_by_label.mean(axis=0)

    data = {
        "feature_idx": np.arange(n_features, dtype=int),
        "split": split_name,
        "position_index_zero_based": int(p_idx),
        "varied_position_one_based": varied_position,
        "dps_a_count": stats["count"][p_idx, :].astype(int),
        "satisfies_all4_dps_a": stats["count"][p_idx, :].astype(int) == int(NUM_LABELS),
        "mean_gap": stats["mean_gap"][p_idx, :],
        "min_gap": stats["min_gap"][p_idx, :],
        "average_matched_preactivation": average_matched,
    }

    for label_id in LABEL_IDS:
        label_id = int(label_id)
        label_name = LABEL_TO_WORD[label_id]
        col = _safe_label_col_name(label_name)
        data[f"matched_preactivation_{col}"] = mean_A[p_idx, label_id, label_id, :]
        data[f"gap_{col}"] = stats["gap"][p_idx, label_id, :]
        data[f"condition_{col}"] = stats["condition"][p_idx, label_id, :]

    df = pd.DataFrame(data)

    if require_all4:
        df = df[df["satisfies_all4_dps_a"]].copy()

    if require_positive_matched:
        df = df[df["average_matched_preactivation"] > 0].copy()

    if df.empty:
        raise RuntimeError(
            "No features passed the requested filters. "
            f"position_index={p_idx}, varied_position={varied_position}, "
            f"split={split_name}, require_all4={require_all4}, "
            f"require_positive_matched={require_positive_matched}."
        )

    df = (
        df.sort_values(
            ["dps_a_count", "mean_gap", "min_gap", "average_matched_preactivation"],
            ascending=[False, False, False, False],
        )
        .head(int(top_n))
        .reset_index(drop=True)
    )
    df.insert(0, "rank", np.arange(1, len(df) + 1, dtype=int))
    return df


def plot_top_dps_a_features(
    feature_df: pd.DataFrame,
    title_prefix: str = "Top DPS-A features",
    filename: str = "top_dps_a_features_by_position",
):
    plot_df = feature_df.copy()
    custom_cols = [
        "feature_idx",
        "dps_a_count",
        "mean_gap",
        "min_gap",
        "average_matched_preactivation",
        "varied_position_one_based",
    ]
    customdata = plot_df[custom_cols].to_numpy()

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=plot_df["rank"],
        y=plot_df["mean_gap"],
        customdata=customdata,
        hovertemplate=(
            "rank=%{x}<br>"
            "feature=%{customdata[0]}<br>"
            "DPS-A count=%{customdata[1]} / " + str(NUM_LABELS) + "<br>"
            "mean gap=%{customdata[2]:.6f}<br>"
            "min gap=%{customdata[3]:.6f}<br>"
            "avg matched preactivation=%{customdata[4]:.6f}<br>"
            "varied position=%{customdata[5]}<extra></extra>"
        ),
        name="Mean DPS-A gap",
    ))

    fig.add_hline(y=0.0, line_width=1)

    split_name = str(plot_df["split"].iloc[0])
    p_idx = int(plot_df["position_index_zero_based"].iloc[0])
    varied_position = int(plot_df["varied_position_one_based"].iloc[0])

    fig.update_layout(
        title=dict(
            text=(
                f"{title_prefix}: top {len(plot_df)} features "
                f"at position index {p_idx} (varied position {varied_position}, {split_name})"
            ),
            x=0.0,
            xanchor="left",
            font=dict(size=24, family=PLOT_FONT),
        ),
        xaxis=dict(
            title=_axis_title_for_todo("Feature rank", size=18),
            tickfont=dict(size=14, family=PLOT_FONT),
        ),
        yaxis=dict(
            title=_axis_title_for_todo("Mean matched − best non-matched preactivation", size=18),
            tickfont=dict(size=14, family=PLOT_FONT),
            zeroline=True,
        ),
        width=1000,
        height=560,
        template="plotly_white",
        font=dict(family=PLOT_FONT, size=14),
        showlegend=False,
    )

    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": filename,
            "width": 1000,
            "height": 560,
            "scale": 4,
        }
    })
    return fig


# Main ToDo 1 table + plot.
top100_dps_a_features_df = build_top_dps_a_features_table(
    position_index=TOP_DPS_A_POSITION_INDEX,
    split_name=TOP_DPS_A_SPLIT,
    top_n=TOP_DPS_A_N,
    require_all4=TOP_DPS_A_REQUIRE_ALL4,
    require_positive_matched=False,
)

print(
    "Top DPS-A features: "
    f"position_index_zero_based={TOP_DPS_A_POSITION_INDEX}, "
    f"varied_position_one_based={int(top100_dps_a_features_df['varied_position_one_based'].iloc[0])}, "
    f"split={TOP_DPS_A_SPLIT}, "
    f"require_all4={TOP_DPS_A_REQUIRE_ALL4}"
)
display(top100_dps_a_features_df)

plot_top_dps_a_features(
    top100_dps_a_features_df,
    title_prefix="Top DPS-A features",
    filename="top100_dps_a_features_position_index_0",
)


## Filled ToDo 2: top-100 DPS-A features with positive average matched preactivation

This repeats the previous ranking and plot after applying the additional filter `average_matched_preactivation > 0`.


In [ ]:
# ============================================================
# ToDo 2 filled: Top-k DPS-A features with positive matched preactivation
# ============================================================

# Uses the same configurable position/split/top-k settings as ToDo 1 by default.
# Change these independently if desired.
TOP_DPS_A_POSITIVE_POSITION_INDEX = TOP_DPS_A_POSITION_INDEX
TOP_DPS_A_POSITIVE_SPLIT = TOP_DPS_A_SPLIT
TOP_DPS_A_POSITIVE_N = TOP_DPS_A_N
TOP_DPS_A_POSITIVE_REQUIRE_ALL4 = TOP_DPS_A_REQUIRE_ALL4


top100_dps_a_features_positive_matched_df = build_top_dps_a_features_table(
    position_index=TOP_DPS_A_POSITIVE_POSITION_INDEX,
    split_name=TOP_DPS_A_POSITIVE_SPLIT,
    top_n=TOP_DPS_A_POSITIVE_N,
    require_all4=TOP_DPS_A_POSITIVE_REQUIRE_ALL4,
    require_positive_matched=True,
)

print(
    "Top DPS-A features with average matched preactivation > 0: "
    f"position_index_zero_based={TOP_DPS_A_POSITIVE_POSITION_INDEX}, "
    f"varied_position_one_based={int(top100_dps_a_features_positive_matched_df['varied_position_one_based'].iloc[0])}, "
    f"split={TOP_DPS_A_POSITIVE_SPLIT}, "
    f"require_all4={TOP_DPS_A_POSITIVE_REQUIRE_ALL4}"
)
display(top100_dps_a_features_positive_matched_df)

plot_top_dps_a_features(
    top100_dps_a_features_positive_matched_df,
    title_prefix="Top DPS-A features with avg matched preactivation > 0",
    filename="top100_dps_a_features_positive_matched_position_index_0",
)


## 8. Holdout position-specificity summaries

These tables and plots compare the selected first-position DPS-A features against two controls:

- `Top-activation control`: features with large absolute mean pre-activation on discovery prompts, excluding selected DPS-A features.
- `Random control`: randomly sampled features, matched in count to the selected DPS-A set and excluding selected DPS-A features.


In [ ]:
# ============================================================
# Feature-group controls and holdout summaries
# ============================================================

rng = np.random.default_rng(int(RANDOM_CONTROL_SEED))
n_select = len(selected_dps_a_features)
all_feature_indices = np.arange(n_features, dtype=int)
available_control_features = np.array([i for i in all_feature_indices if int(i) not in selected_set], dtype=int)
if len(available_control_features) < n_select:
    raise RuntimeError("Not enough non-DPS features to build matched controls.")

# Top-activation controls: matched count, excluding selected DPS-A features.
top_activation_features = (
    feature_discovery_df[~feature_discovery_df["feature_idx"].isin(selected_set)]
    .sort_values("abs_global_mean_preactivation_discovery", ascending=False)
    .head(n_select)["feature_idx"]
    .astype(int)
    .tolist()
)

# Random controls: matched count, excluding selected DPS-A features.
random_control_features = rng.choice(available_control_features, size=n_select, replace=False).astype(int).tolist()

feature_groups = {
    "Discovery DPS-A": selected_dps_a_features,
    "Top-activation control": top_activation_features,
    "Random control": random_control_features,
}

for name, feats in feature_groups.items():
    print(f"{name}: n={len(feats)}; first features={list(map(int, feats[:20]))}")


def _sem_1d(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float).reshape(-1)
    x = x[np.isfinite(x)]
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1) / np.sqrt(len(x)))


def summarize_feature_group(feature_ids: Sequence[int], group_name: str, split_name: str = EVAL_SPLIT) -> pd.DataFrame:
    feature_ids = list(map(int, feature_ids))
    stats = stats_by_split[split_name]
    rows = []
    for p_idx, position in IDX_TO_POSITION.items():
        gap_values = stats["gap"][p_idx, :, :][:, feature_ids].reshape(-1)
        cond_values = stats["condition"][p_idx, :, :][:, feature_ids].astype(float).reshape(-1)
        counts = stats["count"][p_idx, feature_ids]
        rows.append({
            "split": split_name,
            "group": group_name,
            "varied_position": int(position),
            "n_features": int(len(feature_ids)),
            "mean_gap": float(np.mean(gap_values)),
            "sem_gap": _sem_1d(gap_values),
            "condition_rate": float(np.mean(cond_values)),
            "all4_feature_rate": float(np.mean(counts == NUM_LABELS)),
            "num_all4_features": int(np.sum(counts == NUM_LABELS)),
        })
    return pd.DataFrame(rows)


group_summary_df = pd.concat(
    [summarize_feature_group(feats, name, split_name=EVAL_SPLIT) for name, feats in feature_groups.items()],
    ignore_index=True,
)

print("Holdout group summary by varied position:")
display(group_summary_df)

# Position-1 specificity index: gap at position 1 minus mean gap over later positions.
def bootstrap_mean_ci(values: np.ndarray, n_boot: int = 5000, seed: int = 0, ci: float = 95.0) -> Tuple[float, float]:
    values = np.asarray(values, dtype=float).reshape(-1)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    if len(values) == 1:
        return float(values[0]), float(values[0])
    rng_local = np.random.default_rng(int(seed))
    means = np.empty(int(n_boot), dtype=float)
    n = len(values)
    for b in range(int(n_boot)):
        means[b] = np.mean(values[rng_local.integers(0, n, size=n)])
    alpha = (100.0 - float(ci)) / 2.0
    lo, hi = np.percentile(means, [alpha, 100.0 - alpha])
    return float(lo), float(hi)


def specificity_index_for_group(feature_ids: Sequence[int], group_name: str, split_name: str = EVAL_SPLIT) -> Dict[str, Any]:
    feature_ids = list(map(int, feature_ids))
    stats = stats_by_split[split_name]
    if 1 not in POSITION_TO_IDX:
        raise ValueError("Position 1 must be included in POSITIONS_TO_TEST for the specificity index.")
    p1 = POSITION_TO_IDX[1]
    later = [i for i, p in IDX_TO_POSITION.items() if int(p) != 1]
    pos1_values = stats["gap"][p1, :, :][:, feature_ids]  # [label, feature]
    if len(later) == 0:
        later_values = np.full_like(pos1_values, np.nan)
        psi_values = np.full_like(pos1_values, np.nan)
    else:
        later_values = stats["gap"][later, :, :][:, :, feature_ids].mean(axis=0)  # [label, feature]
        psi_values = pos1_values - later_values
    lo, hi = bootstrap_mean_ci(psi_values, seed=SEED + 991)
    return {
        "split": split_name,
        "group": group_name,
        "n_features": int(len(feature_ids)),
        "position1_mean_gap": float(np.nanmean(pos1_values)),
        "later_positions_mean_gap": float(np.nanmean(later_values)),
        "primacy_specificity_index": float(np.nanmean(psi_values)),
        "bootstrap_ci_low": lo,
        "bootstrap_ci_high": hi,
    }


specificity_df = pd.DataFrame([
    specificity_index_for_group(feats, name, split_name=EVAL_SPLIT)
    for name, feats in feature_groups.items()
])
print("Primacy specificity index = mean gap at position 1 − mean gap across positions 2..M:")
display(specificity_df)

# Add selected-feature all-label counts to the all-feature position count table.
selected_position_rows = []
for p_idx, position in IDX_TO_POSITION.items():
    counts = stats_by_split[EVAL_SPLIT]["count"][p_idx, selected_dps_a_features]
    selected_position_rows.append({
        "split": EVAL_SPLIT,
        "varied_position": int(position),
        "selected_num_all4_features": int(np.sum(counts == NUM_LABELS)),
        "selected_all4_feature_rate": float(np.mean(counts == NUM_LABELS)),
    })
selected_position_count_df = pd.DataFrame(selected_position_rows)
position_count_eval_df = (
    position_feature_count_df[position_feature_count_df["split"] == EVAL_SPLIT]
    .merge(selected_position_count_df, on=["split", "varied_position"], how="left")
)
print("All-feature and selected-feature all-label counts on evaluation split:")
display(position_count_eval_df)


## 9. Independent position-wise DPS-A discovery trend

This section answers a slightly different question from the position-specificity plot above. Instead of selecting DPS-A features only from position 1 and evaluating that fixed set across positions, we independently search for DPS-A-like features at each varied position, then compare the number of discovered features and their held-out matched-label gap.

For each position $r$, query label $\ell$, varied label $c$, and feature $f$, we use $A_f^{(r)}(c\to\ell)$ and the DPS-A gap $\Delta_f^{(r)}(\ell)=A_f^{(r)}(\ell\to\ell)-\max_{c\neq\ell}A_f^{(r)}(c\to\ell)$.

A feature is counted as position-$r$ DPS-A-like when it satisfies the matched-max condition for all AGNews labels on the discovery split.

In this variant, all position-wise searches use the wrapped user/assistant prompt format and no prefix label-exclusion constraint is applied.


In [ ]:
# ============================================================
# Independent position-wise DPS-A discovery trend
# ============================================================

# This is different from the earlier Figure 1-style plot:
#   earlier: select DPS-A features once using position 1, then evaluate those same features across positions.
#   here:    independently search for DPS-A-like features at each position r, then compare counts and gaps.

POSITION_TREND_DISCOVERY_SPLIT = "discovery"
POSITION_TREND_EVAL_SPLIT = EVAL_SPLIT
POSITION_TREND_REQUIRE_ALL4 = True
POSITION_TREND_REQUIRE_POSITIVE_AVG_MATCHED = False
POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP = 200  # set None to disable equal-size top-k comparison

PLOT_FONT = globals().get("PLOT_FONT", "Times New Roman, Times, serif")
POSITION_X = [int(p) for p in POSITIONS_TO_TEST]


def _axis_title_position_trend(text: str, size: int = 20):
    return dict(
        text=text,
        font=dict(
            size=size,
            family=PLOT_FONT,
        ),
    )


def _sem_1d_position_trend(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float).reshape(-1)
    x = x[np.isfinite(x)]
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1) / np.sqrt(len(x)))


def build_position_specific_discovery_table(
    p_idx: int,
    split_name: str = POSITION_TREND_DISCOVERY_SPLIT,
    require_all4: bool = POSITION_TREND_REQUIRE_ALL4,
    require_positive_avg_matched: bool = POSITION_TREND_REQUIRE_POSITIVE_AVG_MATCHED,
) -> pd.DataFrame:
    """Rank DPS-A-like features discovered independently at one varied position.

    p_idx is the zero-based index into POSITIONS_TO_TEST / mean_A_by_split arrays.
    """
    p_idx = int(p_idx)
    varied_position = int(IDX_TO_POSITION[p_idx])
    stats = stats_by_split[split_name]
    mean_A = mean_A_by_split[split_name]

    matched_by_label = np.stack(
        [mean_A[p_idx, int(label_id), int(label_id), :] for label_id in LABEL_IDS],
        axis=0,
    )
    average_matched = matched_by_label.mean(axis=0)

    df = pd.DataFrame({
        "feature_idx": np.arange(n_features, dtype=int),
        "discovery_split": split_name,
        "position_index_zero_based": p_idx,
        "varied_position": varied_position,
        "dps_a_count": stats["count"][p_idx, :].astype(int),
        "satisfies_all4_dps_a": stats["count"][p_idx, :].astype(int) == int(NUM_LABELS),
        "discovery_mean_gap": stats["mean_gap"][p_idx, :],
        "discovery_min_gap": stats["min_gap"][p_idx, :],
        "discovery_average_matched_preactivation": average_matched,
    })

    for label_id in LABEL_IDS:
        label_id = int(label_id)
        label_name = re.sub(r"[^0-9a-zA-Z]+", "_", str(LABEL_TO_WORD[label_id]).lower()).strip("_")
        df[f"discovery_gap_{label_name}"] = stats["gap"][p_idx, label_id, :]
        df[f"discovery_condition_{label_name}"] = stats["condition"][p_idx, label_id, :]
        df[f"discovery_matched_preactivation_{label_name}"] = mean_A[p_idx, label_id, label_id, :]

    if require_all4:
        df = df[df["satisfies_all4_dps_a"]].copy()

    if require_positive_avg_matched:
        df = df[df["discovery_average_matched_preactivation"] > 0].copy()

    if df.empty:
        return df.reset_index(drop=True)

    return (
        df.sort_values(
            ["dps_a_count", "discovery_mean_gap", "discovery_min_gap", "discovery_average_matched_preactivation"],
            ascending=[False, False, False, False],
        )
        .reset_index(drop=True)
    )


def evaluate_feature_set_at_position(
    feature_ids: Sequence[int],
    p_idx: int,
    split_name: str = POSITION_TREND_EVAL_SPLIT,
) -> Dict[str, Any]:
    feature_ids = [int(f) for f in feature_ids]
    p_idx = int(p_idx)
    varied_position = int(IDX_TO_POSITION[p_idx])

    if len(feature_ids) == 0:
        return {
            "eval_split": split_name,
            "eval_position_index_zero_based": p_idx,
            "eval_varied_position": varied_position,
            "n_features_evaluated": 0,
            "eval_mean_gap": np.nan,
            "eval_sem_gap": np.nan,
            "eval_condition_rate": np.nan,
            "eval_num_all4_features": 0,
            "eval_all4_feature_rate": np.nan,
        }

    stats = stats_by_split[split_name]
    gap_values = stats["gap"][p_idx, :, :][:, feature_ids].reshape(-1)
    cond_values = stats["condition"][p_idx, :, :][:, feature_ids].astype(float).reshape(-1)
    counts = stats["count"][p_idx, feature_ids]

    return {
        "eval_split": split_name,
        "eval_position_index_zero_based": p_idx,
        "eval_varied_position": varied_position,
        "n_features_evaluated": int(len(feature_ids)),
        "eval_mean_gap": float(np.nanmean(gap_values)),
        "eval_sem_gap": _sem_1d_position_trend(gap_values),
        "eval_condition_rate": float(np.nanmean(cond_values)),
        "eval_num_all4_features": int(np.sum(counts == NUM_LABELS)),
        "eval_all4_feature_rate": float(np.mean(counts == NUM_LABELS)),
    }


# Build independent position-specific discovery groups.
position_specific_discovery_tables = {}
position_specific_feature_groups = {}
position_specific_topk_feature_groups = {}
trend_rows = []

for p_idx, varied_position in IDX_TO_POSITION.items():
    discovery_table = build_position_specific_discovery_table(
        p_idx=p_idx,
        split_name=POSITION_TREND_DISCOVERY_SPLIT,
        require_all4=POSITION_TREND_REQUIRE_ALL4,
        require_positive_avg_matched=POSITION_TREND_REQUIRE_POSITIVE_AVG_MATCHED,
    )
    position_specific_discovery_tables[int(varied_position)] = discovery_table

    all_features_at_position = discovery_table["feature_idx"].astype(int).tolist() if len(discovery_table) else []
    position_specific_feature_groups[int(varied_position)] = all_features_at_position

    if POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP is None:
        topk_features_at_position = all_features_at_position
    else:
        topk_features_at_position = discovery_table.head(int(POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP))["feature_idx"].astype(int).tolist() if len(discovery_table) else []
    position_specific_topk_feature_groups[int(varied_position)] = topk_features_at_position

    eval_all = evaluate_feature_set_at_position(
        all_features_at_position,
        p_idx=p_idx,
        split_name=POSITION_TREND_EVAL_SPLIT,
    )
    eval_topk = evaluate_feature_set_at_position(
        topk_features_at_position,
        p_idx=p_idx,
        split_name=POSITION_TREND_EVAL_SPLIT,
    )

    trend_rows.append({
        "varied_position": int(varied_position),
        "position_index_zero_based": int(p_idx),
        "discovery_split": POSITION_TREND_DISCOVERY_SPLIT,
        "eval_split": POSITION_TREND_EVAL_SPLIT,
        "n_discovery_all4_features": int(len(all_features_at_position)),
        "n_topk_features_for_equalized_gap": int(len(topk_features_at_position)),
        "discovery_mean_gap_all_discovered": float(discovery_table["discovery_mean_gap"].mean()) if len(discovery_table) else np.nan,
        "discovery_min_gap_all_discovered": float(discovery_table["discovery_min_gap"].mean()) if len(discovery_table) else np.nan,
        "holdout_mean_gap_all_discovered": eval_all["eval_mean_gap"],
        "holdout_sem_gap_all_discovered": eval_all["eval_sem_gap"],
        "holdout_condition_rate_all_discovered": eval_all["eval_condition_rate"],
        "holdout_num_all4_among_discovered": eval_all["eval_num_all4_features"],
        "holdout_all4_rate_among_discovered": eval_all["eval_all4_feature_rate"],
        "holdout_mean_gap_topk": eval_topk["eval_mean_gap"],
        "holdout_sem_gap_topk": eval_topk["eval_sem_gap"],
        "holdout_condition_rate_topk": eval_topk["eval_condition_rate"],
    })

positionwise_discovery_trend_df = pd.DataFrame(trend_rows).sort_values("varied_position").reset_index(drop=True)

print("Independent position-wise DPS-A discovery trend")
print("Discovery split:", POSITION_TREND_DISCOVERY_SPLIT)
print("Evaluation split:", POSITION_TREND_EVAL_SPLIT)
print("Require all task-specific DPS-A label conditions:", POSITION_TREND_REQUIRE_ALL4)
print("Require positive avg matched preactivation:", POSITION_TREND_REQUIRE_POSITIVE_AVG_MATCHED)
display(positionwise_discovery_trend_df)


def plot_positionwise_discovery_counts():
    sub = positionwise_discovery_trend_df.sort_values("varied_position")
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=sub["varied_position"],
        y=sub["n_discovery_all4_features"],
        name="Discovered all-label features",
    ))
    fig.add_trace(go.Bar(
        x=sub["varied_position"],
        y=sub["holdout_num_all4_among_discovered"],
        name="Still all-label on holdout",
    ))
    if 1 in POSITION_X:
        fig.add_vrect(
            x0=0.5,
            x1=1.5,
            fillcolor="gray",
            opacity=0.10,
            line_width=0,
            annotation_text="first demo",
            annotation_position="top left",
        )
    fig.update_layout(
        title=dict(
            text="Independent DPS-A feature discovery count by varied position",
            x=0.0,
            xanchor="left",
            font=dict(size=26, family=PLOT_FONT),
        ),
        xaxis=dict(
            title=_axis_title_position_trend("Varied demonstration position", size=20),
            tickmode="array",
            tickvals=POSITION_X,
            ticktext=[str(p) for p in POSITION_X],
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        yaxis=dict(
            title=_axis_title_position_trend("Number of SAE features", size=20),
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        barmode="group",
        width=950,
        height=560,
        template="plotly_white",
        font=dict(family=PLOT_FONT, size=14),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "dps_a_independent_discovery_counts_by_position",
            "width": 950,
            "height": 560,
            "scale": 4,
        }
    })
    return fig


def plot_positionwise_discovery_gap_trend():
    sub = positionwise_discovery_trend_df.sort_values("varied_position")
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sub["varied_position"],
        y=sub["holdout_mean_gap_all_discovered"],
        error_y=dict(
            type="data",
            array=sub["holdout_sem_gap_all_discovered"],
            visible=True,
            width=4,
        ),
        mode="lines+markers",
        name="All discovered features",
    ))
    if POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP is not None:
        fig.add_trace(go.Scatter(
            x=sub["varied_position"],
            y=sub["holdout_mean_gap_topk"],
            error_y=dict(
                type="data",
                array=sub["holdout_sem_gap_topk"],
                visible=True,
                width=4,
            ),
            mode="lines+markers",
            name=f"Top-{POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP} per position",
        ))
    if 1 in POSITION_X:
        fig.add_vrect(
            x0=0.5,
            x1=1.5,
            fillcolor="gray",
            opacity=0.10,
            line_width=0,
            annotation_text="first demo",
            annotation_position="top left",
        )
    fig.update_layout(
        title=dict(
            text="Independent DPS-A feature groups: held-out matched gap by discovery position",
            x=0.0,
            xanchor="left",
            font=dict(size=26, family=PLOT_FONT),
        ),
        xaxis=dict(
            title=_axis_title_position_trend("Discovery / varied demonstration position", size=20),
            tickmode="array",
            tickvals=POSITION_X,
            ticktext=[str(p) for p in POSITION_X],
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        yaxis=dict(
            title=_axis_title_position_trend("Held-out matched − best non-matched preactivation", size=20),
            zeroline=True,
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        width=950,
        height=560,
        template="plotly_white",
        font=dict(family=PLOT_FONT, size=14),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "dps_a_independent_discovery_gap_by_position",
            "width": 950,
            "height": 560,
            "scale": 4,
        }
    })
    return fig


def build_position_group_cross_eval_df(use_topk: bool = False) -> pd.DataFrame:
    rows = []
    group_map = position_specific_topk_feature_groups if use_topk else position_specific_feature_groups
    for discovery_position, feature_ids in group_map.items():
        for eval_p_idx, eval_position in IDX_TO_POSITION.items():
            metrics = evaluate_feature_set_at_position(
                feature_ids,
                p_idx=eval_p_idx,
                split_name=POSITION_TREND_EVAL_SPLIT,
            )
            rows.append({
                "discovery_position": int(discovery_position),
                "eval_position": int(eval_position),
                "n_features": int(len(feature_ids)),
                "mean_gap": metrics["eval_mean_gap"],
                "condition_rate": metrics["eval_condition_rate"],
                "num_all4_features": metrics["eval_num_all4_features"],
                "use_topk": bool(use_topk),
            })
    return pd.DataFrame(rows)


position_group_cross_eval_df = build_position_group_cross_eval_df(use_topk=False)
position_group_cross_eval_topk_df = (
    build_position_group_cross_eval_df(use_topk=True)
    if POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP is not None
    else pd.DataFrame()
)

print("Cross-evaluation matrix rows: independently discovered group at one position, evaluated at every position")
display(position_group_cross_eval_df.head(20))


def plot_position_group_cross_eval_heatmap(use_topk: bool = False):
    df = position_group_cross_eval_topk_df if use_topk else position_group_cross_eval_df
    if df.empty:
        print("No cross-evaluation data to plot.")
        return None

    positions = [int(p) for p in POSITION_X]
    z = np.full((len(positions), len(positions)), np.nan, dtype=float)
    for i, discovery_position in enumerate(positions):
        for j, eval_position in enumerate(positions):
            vals = df[
                (df["discovery_position"] == int(discovery_position))
                & (df["eval_position"] == int(eval_position))
            ]["mean_gap"]
            z[i, j] = float(vals.iloc[0]) if len(vals) else np.nan

    text = np.where(np.isnan(z), "", np.char.mod("%.3f", z))
    title_suffix = (
        f"top-{POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP}" if use_topk else "all discovered"
    )

    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=[str(p) for p in positions],
        y=[str(p) for p in positions],
        text=text,
        texttemplate="%{text}",
        colorscale="RdBu",
        zmid=0.0,
        colorbar=dict(title=dict(text="Mean gap")),
        hovertemplate=(
            "Discovery position: %{y}<br>"
            "Evaluation position: %{x}<br>"
            "Mean held-out gap: %{z:.5f}<extra></extra>"
        ),
    ))
    fig.update_layout(
        title=dict(
            text=f"Position-specific DPS-A groups cross-evaluated across positions ({title_suffix})",
            x=0.0,
            xanchor="left",
            font=dict(size=24, family=PLOT_FONT),
        ),
        xaxis=dict(
            title=_axis_title_position_trend("Evaluation varied position", size=20),
            categoryorder="array",
            categoryarray=[str(p) for p in positions],
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        yaxis=dict(
            title=_axis_title_position_trend("Discovery varied position", size=20),
            categoryorder="array",
            categoryarray=[str(p) for p in positions],
            autorange="reversed",
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        width=760,
        height=680,
        template="plotly_white",
        font=dict(family=PLOT_FONT, size=14),
    )
    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"dps_a_position_group_cross_eval_{'topk' if use_topk else 'all'}",
            "width": 760,
            "height": 680,
            "scale": 4,
        }
    })
    return fig


plot_positionwise_discovery_counts()
plot_positionwise_discovery_gap_trend()
plot_position_group_cross_eval_heatmap(use_topk=False)
if POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP is not None:
    plot_position_group_cross_eval_heatmap(use_topk=True)


## 9. Visualizations

The main reviewer-facing figures are:

1. Mean matched-vs-best-alternative DPS-A gap by varied position.
2. DPS-A condition satisfaction rate by varied position.
3. Number of all-4 DPS-A-like features by varied position.
4. Label-resolved heatmap of the selected DPS-A population's matched gaps.
5. Optional feature-level position matrices for the strongest selected feature.


In [ ]:
# ============================================================
# Visualization helpers
# ============================================================

from typing import Optional, List
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

PLOT_FONT = "Times New Roman, Times, serif"
POSITION_X = [int(p) for p in POSITIONS_TO_TEST]


def axis_title(text: str, size: int = 20):
    return dict(
        text=text,
        font=dict(
            size=size,
            family=PLOT_FONT,
        ),
    )


def plot_group_gap_by_position():
    fig = go.Figure()

    group_order = [
        "Discovery DPS-A",
        "Top-activation control",
        "Random control",
    ]

    for group_name in group_order:
        sub = (
            group_summary_df[group_summary_df["group"] == group_name]
            .sort_values("varied_position")
        )

        fig.add_trace(go.Scatter(
            x=sub["varied_position"],
            y=sub["mean_gap"],
            error_y=dict(
                type="data",
                array=sub["sem_gap"],
                visible=True,
                width=4,
            ),
            mode="lines+markers",
            name=group_name,
        ))

    if 1 in POSITION_X:
        fig.add_vrect(
            x0=0.5,
            x1=1.5,
            fillcolor="gray",
            opacity=0.12,
            line_width=0,
            annotation_text="first demo",
            annotation_position="top left",
        )

    fig.update_layout(
        title=dict(
            text="First-position specificity of discovered DPS-A features",
            x=0.0,
            xanchor="left",
            font=dict(
                size=26,
                family=PLOT_FONT,
            ),
        ),
        xaxis=dict(
            title=axis_title("Varied demonstration position", size=20),
            tickmode="array",
            tickvals=POSITION_X,
            ticktext=[str(p) for p in POSITION_X],
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        yaxis=dict(
            title=axis_title(
                "Matched − best non-matched mean pre-activation",
                size=20,
            ),
            zeroline=True,
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        width=950,
        height=560,
        template="plotly_white",
        font=dict(
            family=PLOT_FONT,
            size=14,
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0.0,
        ),
    )

    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "dps_a_position_specificity_gap",
            "width": 950,
            "height": 560,
            "scale": 4,
        }
    })

    return fig


def plot_condition_rate_by_position():
    fig = go.Figure()

    group_order = [
        "Discovery DPS-A",
        "Top-activation control",
        "Random control",
    ]

    for group_name in group_order:
        sub = (
            group_summary_df[group_summary_df["group"] == group_name]
            .sort_values("varied_position")
        )

        fig.add_trace(go.Scatter(
            x=sub["varied_position"],
            y=sub["condition_rate"],
            mode="lines+markers",
            name=group_name,
        ))

    if 1 in POSITION_X:
        fig.add_vrect(
            x0=0.5,
            x1=1.5,
            fillcolor="gray",
            opacity=0.12,
            line_width=0,
        )

    fig.update_layout(
        title=dict(
            text="DPS-A condition satisfaction rate by varied position",
            x=0.0,
            xanchor="left",
            font=dict(
                size=26,
                family=PLOT_FONT,
            ),
        ),
        xaxis=dict(
            title=axis_title("Varied demonstration position", size=20),
            tickmode="array",
            tickvals=POSITION_X,
            ticktext=[str(p) for p in POSITION_X],
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        yaxis=dict(
            title=axis_title(
                "Fraction of label-feature conditions satisfied",
                size=20,
            ),
            range=[0, 1.02],
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        width=950,
        height=560,
        template="plotly_white",
        font=dict(
            family=PLOT_FONT,
            size=14,
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0.0,
        ),
    )

    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "dps_a_position_specificity_condition_rate",
            "width": 950,
            "height": 560,
            "scale": 4,
        }
    })

    return fig


def plot_all4_counts_by_position():
    sub = position_count_eval_df.sort_values("varied_position")

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=sub["varied_position"],
        y=sub["num_all4_dps_a_features"],
        name="All SAE features satisfying all 4 conditions",
    ))

    fig.add_trace(go.Bar(
        x=sub["varied_position"],
        y=sub["selected_num_all4_features"],
        name="Selected discovery DPS-A features satisfying all 4",
    ))

    if 1 in POSITION_X:
        fig.add_vrect(
            x0=0.5,
            x1=1.5,
            fillcolor="gray",
            opacity=0.10,
            line_width=0,
        )

    fig.update_layout(
        title=dict(
            text=f"All-label DPS-A-like feature counts by varied position ({EVAL_SPLIT})",
            x=0.0,
            xanchor="left",
            font=dict(
                size=26,
                family=PLOT_FONT,
            ),
        ),
        xaxis=dict(
            title=axis_title("Varied demonstration position", size=20),
            tickmode="array",
            tickvals=POSITION_X,
            ticktext=[str(p) for p in POSITION_X],
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        yaxis=dict(
            title=axis_title("Number of features", size=20),
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        barmode="group",
        width=950,
        height=560,
        template="plotly_white",
        font=dict(
            family=PLOT_FONT,
            size=14,
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0.0,
        ),
    )

    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "dps_a_all4_counts_by_position",
            "width": 950,
            "height": 560,
            "scale": 4,
        }
    })

    return fig


def plot_selected_gap_heatmap():
    stats = stats_by_split[EVAL_SPLIT]

    # Shape: [position, query_label]
    z = stats["gap"][:, :, selected_dps_a_features].mean(axis=2)
    text = np.char.mod("%.3f", z)

    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=[LABEL_TO_WORD[i] for i in LABEL_IDS],
        y=[str(p) for p in POSITIONS_TO_TEST],
        text=text,
        texttemplate="%{text}",
        colorscale="RdBu",
        zmid=0.0,
        colorbar=dict(
            title=dict(
                text="Matched gap",
            ),
        ),
        hovertemplate=(
            "Varied position: %{y}<br>"
            "Test query label: %{x}<br>"
            "Mean matched gap: %{z:.5f}<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=dict(
            text=f"Selected discovery DPS-A features: label-resolved matched gaps ({EVAL_SPLIT})",
            x=0.0,
            xanchor="left",
            font=dict(
                size=24,
                family=PLOT_FONT,
            ),
        ),
        xaxis=dict(
            title=axis_title("Test query label", size=20),
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        yaxis=dict(
            title=axis_title("Varied demonstration position", size=20),
            categoryorder="array",
            categoryarray=[str(p) for p in POSITIONS_TO_TEST],
            autorange="reversed",
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        width=780,
        height=640,
        template="plotly_white",
        font=dict(
            family=PLOT_FONT,
            size=14,
        ),
    )

    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "dps_a_selected_label_resolved_gap_heatmap",
            "width": 780,
            "height": 640,
            "scale": 4,
        }
    })

    return fig


def representative_selected_feature() -> int:
    # Pick selected feature with largest discovery position-1 mean gap.
    sub = feature_discovery_df[
        feature_discovery_df["feature_idx"].isin(selected_dps_a_features)
    ]

    return int(
        sub
        .sort_values("discovery_pos1_mean_gap", ascending=False)
        .iloc[0]["feature_idx"]
    )


def plot_feature_position_matrices(
    feature_idx: Optional[int] = None,
    positions_to_show: Optional[List[int]] = None,
):
    if feature_idx is None:
        feature_idx = representative_selected_feature()

    feature_idx = int(feature_idx)

    if positions_to_show is None:
        candidate_positions = [
            1,
            2,
            int(round((int(NUM_DEMOS_TOTAL) + 1) / 2)),
            int(NUM_DEMOS_TOTAL),
        ]

        positions_to_show = []

        for p in candidate_positions + list(POSITIONS_TO_TEST):
            p = int(p)

            if p in POSITION_TO_IDX and p not in positions_to_show:
                positions_to_show.append(p)

            if len(positions_to_show) >= 4:
                break

    positions_to_show = [
        int(p)
        for p in positions_to_show
        if int(p) in POSITION_TO_IDX
    ]

    if len(positions_to_show) == 0:
        raise ValueError(
            "No valid positions_to_show were found in POSITION_TO_IDX."
        )

    label_order = [LABEL_TO_WORD[i] for i in LABEL_IDS]
    n_cols = len(positions_to_show)

    fig = make_subplots(
        rows=1,
        cols=n_cols,
        subplot_titles=[
            f"Varied position {p}"
            for p in positions_to_show
        ],
        specs=[
            [{"type": "heatmap"} for _ in range(n_cols)]
        ],
        horizontal_spacing=0.05,
    )

    z_values = []

    for p in positions_to_show:
        z_values.append(
            mean_A_by_split[EVAL_SPLIT][
                POSITION_TO_IDX[p],
                :,
                :,
                feature_idx,
            ].T
        )

    z_all = np.concatenate([
        z.reshape(-1)
        for z in z_values
    ])

    finite = z_all[np.isfinite(z_all)]
    z_abs = float(np.max(np.abs(finite))) if len(finite) else 1.0

    for col, (p, z) in enumerate(zip(positions_to_show, z_values), start=1):
        fig.add_trace(
            go.Heatmap(
                z=z,
                x=label_order,
                y=label_order,
                text=np.char.mod("%.3f", z),
                texttemplate="%{text}",
                colorscale="RdBu",
                zmid=0.0,
                zmin=-z_abs,
                zmax=z_abs,
                colorbar=(
                    dict(
                        title=dict(
                            text="Mean pre-act",
                        ),
                    )
                    if col == n_cols
                    else None
                ),
                showscale=(col == n_cols),
                hovertemplate=(
                    "Varied label: %{x}<br>"
                    "Test query label: %{y}<br>"
                    "Mean pre-act: %{z:.5f}<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(
            title=dict(
                text="Varied label",
                font=dict(
                    size=14,
                    family=PLOT_FONT,
                ),
            ),
            tickangle=-35,
            tickfont=dict(
                size=12,
                family=PLOT_FONT,
            ),
            row=1,
            col=col,
        )

        fig.update_yaxes(
            title=dict(
                text="Test label" if col == 1 else "",
                font=dict(
                    size=14,
                    family=PLOT_FONT,
                ),
            ),
            autorange="reversed",
            tickfont=dict(
                size=12,
                family=PLOT_FONT,
            ),
            row=1,
            col=col,
        )

    fig.update_layout(
        title=dict(
            text=f"Representative selected DPS-A feature {feature_idx}: position matrices ({EVAL_SPLIT})",
            x=0.0,
            xanchor="left",
            font=dict(
                size=22,
                family=PLOT_FONT,
            ),
        ),
        width=max(360 * n_cols, 800),
        height=520,
        template="plotly_white",
        font=dict(
            family=PLOT_FONT,
            size=13,
        ),
    )

    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"dps_a_feature_{feature_idx}_position_matrices",
            "width": max(360 * n_cols, 800),
            "height": 520,
            "scale": 4,
        }
    })

    return fig


plot_group_gap_by_position()
plot_condition_rate_by_position()
plot_all4_counts_by_position()
plot_selected_gap_heatmap()
plot_feature_position_matrices()

## 10. Save compact CSV summaries for cross-task aggregation


In [ ]:
# ============================================================
# Save compact summaries for cross-task aggregation
# ============================================================

SAVE_CSV_SUMMARIES = True
SUMMARY_DIR = CROSS_TASK_SUMMARY_ROOT / TASK_KEY
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)


def build_cross_task_all_feature_scores_df() -> pd.DataFrame:
    """One row per (task, varied_position, feature) with discovery and holdout DPS-A statistics.

    This is the main artifact consumed by the final cross-task aggregation notebook.
    It intentionally saves all features, not only the features passing the all-label DPS-A filter,
    so the final notebook can count features satisfying AGNews+TREC+Yahoo jointly.
    """
    rows = []
    discovery_stats = stats_by_split["discovery"]
    holdout_stats = stats_by_split[EVAL_SPLIT]
    discovery_mean_A = mean_A_by_split["discovery"]
    holdout_mean_A = mean_A_by_split[EVAL_SPLIT]

    for p_idx, varied_position in IDX_TO_POSITION.items():
        p_idx = int(p_idx)
        varied_position = int(varied_position)

        disc_matched = np.stack(
            [discovery_mean_A[p_idx, int(label_id), int(label_id), :] for label_id in LABEL_IDS],
            axis=0,
        ).mean(axis=0)
        hold_matched = np.stack(
            [holdout_mean_A[p_idx, int(label_id), int(label_id), :] for label_id in LABEL_IDS],
            axis=0,
        ).mean(axis=0)

        disc_count = discovery_stats["count"][p_idx, :].astype(int)
        hold_count = holdout_stats["count"][p_idx, :].astype(int)
        disc_mean_gap = discovery_stats["mean_gap"][p_idx, :].astype(float)
        hold_mean_gap = holdout_stats["mean_gap"][p_idx, :].astype(float)
        disc_min_gap = discovery_stats["min_gap"][p_idx, :].astype(float)
        hold_min_gap = holdout_stats["min_gap"][p_idx, :].astype(float)

        for feature_idx in range(int(n_features)):
            rows.append({
                "task_key": TASK_KEY,
                "task_display_name": TASK_DISPLAY_NAME,
                "dataset_name": DATASET_NAME,
                "model_short_name": MODEL_SHORT_NAME,
                "target_layer": int(TARGET_LAYER),
                "prompt_format_name": PROMPT_FORMAT_NAME,
                "last_k": json.dumps(None if LAST_K is None else [int(x) for x in normalize_last_k_spec(LAST_K)]),
                "num_labels": int(NUM_LABELS),
                "varied_position": varied_position,
                "position_index_zero_based": p_idx,
                "feature_idx": int(feature_idx),
                "discovery_dps_a_count": int(disc_count[feature_idx]),
                "discovery_satisfies_all_labels": bool(disc_count[feature_idx] == int(NUM_LABELS)),
                "discovery_mean_gap": float(disc_mean_gap[feature_idx]),
                "discovery_gap_sum": float(disc_mean_gap[feature_idx] * int(NUM_LABELS)),
                "discovery_min_gap": float(disc_min_gap[feature_idx]),
                "discovery_average_matched_preactivation": float(disc_matched[feature_idx]),
                "holdout_dps_a_count": int(hold_count[feature_idx]),
                "holdout_satisfies_all_labels": bool(hold_count[feature_idx] == int(NUM_LABELS)),
                "holdout_mean_gap": float(hold_mean_gap[feature_idx]),
                "holdout_gap_sum": float(hold_mean_gap[feature_idx] * int(NUM_LABELS)),
                "holdout_min_gap": float(hold_min_gap[feature_idx]),
                "holdout_average_matched_preactivation": float(hold_matched[feature_idx]),
            })

    return pd.DataFrame(rows)


cross_task_all_feature_scores_df = build_cross_task_all_feature_scores_df()

# Stable "latest" files used by the final aggregation notebook.
all_feature_scores_latest_path = SUMMARY_DIR / "all_feature_scores_latest.csv.gz"
positionwise_trend_latest_path = SUMMARY_DIR / "positionwise_discovery_trend_latest.csv"
cross_eval_latest_path = SUMMARY_DIR / "position_group_cross_eval_latest.csv"
cross_eval_topk_latest_path = SUMMARY_DIR / "position_group_cross_eval_topk_latest.csv"
feature_discovery_latest_path = SUMMARY_DIR / "feature_discovery_pos1_latest.csv"
position_feature_count_latest_path = SUMMARY_DIR / "position_feature_count_latest.csv"
group_summary_latest_path = SUMMARY_DIR / "group_summary_latest.csv"
specificity_latest_path = SUMMARY_DIR / "specificity_latest.csv"
plan_latest_path = SUMMARY_DIR / "position_scan_plan_latest.csv"
run_meta_latest_path = SUMMARY_DIR / "run_meta_latest.json"

# Fingerprinted files for archival reproducibility.
all_feature_scores_fp_path = SUMMARY_DIR / f"all_feature_scores_{RUN_FINGERPRINT}.csv.gz"
positionwise_trend_fp_path = SUMMARY_DIR / f"positionwise_discovery_trend_{RUN_FINGERPRINT}.csv"

if SAVE_CSV_SUMMARIES:
    cross_task_all_feature_scores_df.to_csv(all_feature_scores_latest_path, index=False, compression="gzip")
    cross_task_all_feature_scores_df.to_csv(all_feature_scores_fp_path, index=False, compression="gzip")
    positionwise_discovery_trend_df.to_csv(positionwise_trend_latest_path, index=False)
    positionwise_discovery_trend_df.to_csv(positionwise_trend_fp_path, index=False)
    position_group_cross_eval_df.to_csv(cross_eval_latest_path, index=False)
    if not position_group_cross_eval_topk_df.empty:
        position_group_cross_eval_topk_df.to_csv(cross_eval_topk_latest_path, index=False)
    feature_discovery_df.to_csv(feature_discovery_latest_path, index=False)
    position_feature_count_df.to_csv(position_feature_count_latest_path, index=False)
    group_summary_df.to_csv(group_summary_latest_path, index=False)
    specificity_df.to_csv(specificity_latest_path, index=False)
    position_scan_plan_df.to_csv(plan_latest_path, index=False)

    run_meta_to_save = dict(run_meta)
    run_meta_to_save.update({
        "task_key": TASK_KEY,
        "task_display_name": TASK_DISPLAY_NAME,
        "dataset_name": DATASET_NAME,
        "summary_dir": str(SUMMARY_DIR),
        "all_feature_scores_latest_path": str(all_feature_scores_latest_path),
        "positionwise_trend_latest_path": str(positionwise_trend_latest_path),
        "num_rows_all_feature_scores": int(len(cross_task_all_feature_scores_df)),
    })
    with open(run_meta_latest_path, "w") as f:
        json.dump(run_meta_to_save, f, indent=2, sort_keys=True)

    print("Saved cross-task aggregation artifacts:")
    print(" ", all_feature_scores_latest_path)
    print(" ", positionwise_trend_latest_path)
    print(" ", run_meta_latest_path)
    print("Rows in all-feature score table:", len(cross_task_all_feature_scores_df))
    display(cross_task_all_feature_scores_df.head())
else:
    print("SAVE_CSV_SUMMARIES=False; no CSV files written.")


## 11. Interpretation guide

This notebook performs two related analyses for **TREC**:

1. It freezes a position-1-discovered DPS-A feature set and evaluates that same set across varied positions.
2. It independently discovers DPS-A-like features at each varied position and evaluates each group on held-out prompts.

The saved file `./dps_positionwise_wrapped_outputs/trec/all_feature_scores_latest.csv.gz` contains one row per `(task, varied_position, feature)` and is consumed by the final cross-task aggregation notebook.
